## Liberaries and data settup

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import time
import os
import os
import tensorflow as tf
from tensorflow.keras import mixed_precision
import gc
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import scipy.linalg
import math
import os, gc, random
import matplotlib.pyplot as plt
import json
import pickle
import time
from datetime import datetime

tf.keras.mixed_precision.set_global_policy('float32')

# Note: Your model will now output float16. 
# Ensure your last layer is float32 for stability:
# self.out = tf.keras.layers.Dense(num_classes, dtype='float32')

# Standard Conda path for libdevice
conda_cuda_path = os.path.join(os.environ.get('CONDA_PREFIX', ''), "Library", "bin")
if os.path.exists(conda_cuda_path):
    os.environ['XLA_FLAGS'] = f"--xla_gpu_cuda_data_dir={conda_cuda_path}"
    print(f"XLA Path set to: {conda_cuda_path}")

# EMERGENCY BYPASS: If it still fails, disable JIT for now. 
# It will be slightly slower, but it will NOT crash.
USE_JIT = False

import os
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir="C:/Users/caspe/anaconda3/envs/SPIKEDETEC/Library/bin"'

In [ ]:
# 1. Load MNIST
(x_all, y_all), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 2. Flatten and Normalize (Standard step)
x_all = x_all.reshape(-1, 784).astype('uint8') / 255.0
x_test = x_test.reshape(-1, 784).astype('uint8') / 255.0

# 3. Create FIXED Permutation (The "p" in pMNIST)
rng = np.random.RandomState(42)
perm = rng.permutation(784)

x_all = x_all[:, perm]
x_test = x_test[:, perm]

# --- NEW: Train/Val Split (90% Train, 10% Val) ---
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(
    x_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)

# 4. Reshape to [Batch, Time, Channels] for your RNN
x_train = x_train[:, :, np.newaxis]
x_val   = x_val[:, :, np.newaxis]
x_test  = x_test[:, :, np.newaxis]

# 5. Labels to int32
y_train = y_train.astype('int32')
y_val   = y_val.astype('int32')
y_test  = y_test.astype('int32')

# 6. Build datasets
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000).batch(256)
val_ds   = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(256)
test_ds  = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(256)

print(f"pMNIST Ready. Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

## Clean environment and drift calc


In [ ]:
import tensorflow as tf
# --- 1. CONFIGURATION ---


# Forcefully disable the strict determinism flag at the TF level
try:
    tf.config.experimental.enable_op_determinism(False)
    print("Success: Strict determinism disabled.")
except:
    print("Warning: Could not toggle flag. If Phase 1 fails, restart kernel.")
def reset_seeds():
    import os
    import gc
    import random
    import numpy as np

    # 1. Clear session
    tf.keras.backend.clear_session()
    
    # 2. Disable strict determinism to allow GPU kernels to run
    os.environ['TF_DETERMINISTIC_OPS'] = '0' 
    
    # 3. Hard-lock all seeds
    os.environ['PYTHONHASHSEED'] = str(42)
    random.seed(42)
    np.random.seed(42)
    tf.random.set_seed(42)
    tf.keras.utils.set_random_seed(42) 
    
    gc.collect()
    print("Environment Reset: Seeds locked at 42. Strict determinism disabled.")

## Variables etc

## Parameter count

In [ ]:
def print_param_report(hidden_size, input_dim=1, num_classes=10):
    """
    Calculates and prints a detailed breakdown of parameters for the 
    OscillatingResonator architecture.
    """
    # RNN Layer 1: Inputs from data
    rnn1_params = (input_dim * hidden_size) + (hidden_size * hidden_size) + hidden_size
    
    # RNN Layer 2 & 3: Inputs from previous hidden layer
    rnn_mid_params = (hidden_size * hidden_size) + (hidden_size * hidden_size) + hidden_size
    
    # Final Dense Layer
    dense_params = (hidden_size * num_classes) + num_classes
    
    total = rnn1_params + (2 * rnn_mid_params) + dense_params
    
    print(f"--- PARAMETER COUNT REPORT (Hidden: {hidden_size}) ---")
    print(f"RNN Layer 1: {rnn1_params:>8,}")
    print(f"RNN Layer 2: {rnn_mid_params:>8,}")
    print(f"RNN Layer 3: {rnn_mid_params:>8,}")
    print(f"Dense Out:   {dense_params:>8,}")
    print("-" * 35)
    print(f"TOTAL PARAMS: {total:>8,}")
    print(f"Estimated Memory: {total * 4 / 1024:.2f} KB (float32)")
    print("=" * 35 + "\n")




In [ ]:
DATA_PERCENT = 0.01
BATCH_SIZE = 4 * 64
EPOCHS = 2         
HIDDEN = 16 
BASE_STRENGTH = 0.22 
PERIOD = 784/4 
LAMBDA_SLOW = 0.015  
JITTER_SCALE = 0.65 
REST_BASELINE = 1.0
LEARNING_RATE = 8e-4
H_SCALE = [0.94, 0.06]    


# --- 1.1 AUTO-GENERATED SESSION NAME ---
# Creates a name like: RES_H32_S0.40_J1.15_T123456
now = datetime.now()
readable_ts = now.strftime("%y%m%d_%H%M") 
SESSION_ID = f"RES_H{HIDDEN}_S{BASE_STRENGTH}_J{JITTER_SCALE}_{readable_ts}"
print(f" SESSION INITIALIZED: {SESSION_ID}")
print_param_report(HIDDEN)

##  Oscillatory global field block 

In [ ]:




# --- 2. THE JITTERED CELL ---
class JitteredFeedbackCell(tf.keras.layers.Layer):
    def __init__(self, units=16, strength=0.0, period=256.0, lambda_slow=0.05, mode="active", **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = [units, units, 1] 
        self.strength = strength 
        self.period = period
        self.lambda_slow = lambda_slow
        self.mode = mode

    def build(self, input_shape):
        self.w_in = self.add_weight(shape=(input_shape[-1], self.units), name="w_in", initializer="glorot_uniform")
        self.w_rec = self.add_weight(shape=(self.units, self.units), name="w_rec",
                                     initializer=tf.keras.initializers.Orthogonal(gain=1.0))
        self.bias = self.add_weight(shape=(self.units,), name="bias", initializer="zeros")

    def call(self, inputs, states):
        prev_h, prev_G, prev_phase = states
        half = self.units // 2
        raw_signal = tf.concat([prev_h[:, half:], prev_h[:, :half]], axis=1)
        source_signal = tf.stop_gradient(raw_signal) if self.mode == "probe" else raw_signal
        new_G = (1.0 - self.lambda_slow) * prev_G + self.lambda_slow * source_signal
        G_mean = tf.reduce_mean(new_G, axis=-1, keepdims=True)
        G_std = tf.math.reduce_std(new_G, axis=-1, keepdims=True) + 1e-6
        G_norm = (new_G - G_mean) / G_std
        new_phase = prev_phase + (2.0 * math.pi / self.period)
        oscillator = tf.math.sin(new_phase)
        
        if self.mode == "active":
            bias_signal = tf.reduce_mean(source_signal, axis=-1, keepdims=True) - 0.1
            combined_signal = oscillator + (JITTER_SCALE * bias_signal)
        else:
            combined_signal = oscillator

        current_strength = self.strength * combined_signal if self.mode != "passive" else 0.0
        field_effect = REST_BASELINE + (current_strength * tf.tanh(G_norm))
        z = (tf.matmul(inputs, self.w_in) + tf.matmul(prev_h, self.w_rec) + self.bias) * field_effect
        h = (H_SCALE[0] * prev_h) + (H_SCALE[1] * tf.nn.elu(z))
        h = tf.clip_by_value(h, -20.0, 20.0)
        return h, [h, new_G, new_phase]

class OscillatingResonator(tf.keras.Model):
    def __init__(self, hidden=16, num_classes=10, strength=0.0, mode="active"):
        super().__init__()
        self.cell_ref = JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode)
        self.rnn1 = tf.keras.layers.RNN(self.cell_ref, return_sequences=True)
        self.rnn2 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.rnn3 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.out = tf.keras.layers.Dense(num_classes)

    def call(self, x, training=False):
        h1 = self.rnn1(x, training=training)
        h2 = self.rnn2(h1, training=training)
        h3 = self.rnn3(h2, training=training)
        return self.out(h3[:, -1, :]), h3

# --- 3. UPDATED LOGGING UTILITY ---
def print_history_summary(history, model, model_name="Model", test_acc=None):
    cell = model.rnn1.cell
    total_params = model.count_params()
    
    print(f"\n DATA LOG: {model_name}")
    print(f" CONFIG: Hidden: {cell.units} | Params: {total_params:,} | F-Wgt: {cell.strength:.4f} | "
          f"L-Slow (τ): {cell.lambda_slow:.3f} | Jitter: {JITTER_SCALE:.2f} | Period: {cell.period:.1f} | Data: {DATA_PERCENT*100:.2f}%")
    print("="*145)
    header = f"{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6} | {'F-Wgt':<6}"
    print(header)
    print("-" * 145)

    for i in range(len(history['loss'])):
        m = history['hidden_metrics'][i]
        print(f"{i+1:<6} | {history['loss'][i]:<7.3f} | {history['acc'][i]*100:<8.2f} | "
              f"{m['effective_rank']:<6.2f} | {m['synchrony']:<6.3f} | {m['entropy']:<6.2f} | "
              f"{m['a_corr']:<7.3f} | {m['interference']:<6.3f} | {cell.strength:<6.3f}")
    
    print("-" * 145)
    test_str = f"{test_acc*100:.2f}%" if test_acc is not None else "N/A"
    print(f" FINAL PERFORMANCE: Validation Acc: {history['acc'][-1]*100:.2f}% | TEST ACCURACY: {test_str}")
    print("="*145 + "\n")

# --- 4. TRAINING PHASE WITH SNAPSHOT & LIVE LOGS ---
# --- UPDATED TRAINING PHASE ---
def train_phase(model, train_data, val_data, epochs=3, name="model"):
    current_lr = LEARNING_RATE
    optimizer = tf.keras.optimizers.Adam(learning_rate=current_lr)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    history = {"loss": [], "acc": [], "hidden_metrics": []}
    best_val_acc = 0.0

    @tf.function
    def train_step(x, y, lr):
        optimizer.learning_rate.assign(lr)
        with tf.GradientTape() as tape:
            logits, _ = model(x, training=True)
            loss_v = loss_fn(y, logits)
        grads = tape.gradient(loss_v, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss_v, logits

    print(f"\n{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7}")
    print("-" * 75)

    for epoch in range(epochs):
        acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
        epoch_loss = []
        pbar = tqdm(train_data, desc=f"EPOCH {epoch+1}/{epochs}", leave=False)
        
        for x_b, y_b in pbar:
            loss_v, logits = train_step(x_b, y_b, current_lr)
            acc_metric.update_state(y_b, logits)
            epoch_loss.append(float(loss_v))
            pbar.set_postfix({"loss": f"{np.mean(epoch_loss):.4f}", "acc": f"{acc_metric.result():.2%}"})

        # --- VALIDATION SNAPSHOT ---
        for x_v, y_v in val_data.take(1):
            logits_v, h_seq_v = model(x_v, training=False)
            val_acc = np.mean(np.argmax(logits_v.numpy(), axis=-1) == y_v.numpy())
            
            # 1. SAVE BEST WEIGHTS
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                model.save_weights(f"best_{name}_{SESSION_ID}.weights.h5")
                status_msg = f" (New Best!)"
            else:
                status_msg = ""
            
            # Metric Calculations
            h_final = h_seq_v.numpy()[:, -1, :]
            s = scipy.linalg.svdvals(h_final) + 1e-12
            p_rank = s / (np.sum(s) + 1e-10)
            eff_rank = np.exp(-np.sum(p_rank * np.log(p_rank + 1e-10)))
            
            counts, _ = np.histogram(h_final, bins=50)
            p_ent = counts / (h_final.size + 1e-10) 
            entropy_val = -np.sum(p_ent * np.log2(p_ent + 1e-10))
            
            sync_val = (np.sum(np.abs(np.corrcoef(h_final.T + 1e-8))) - HIDDEN) / (HIDDEN**2 - HIDDEN)
            acorr_val = np.mean(np.abs(np.corrcoef(h_seq_v.numpy()[0].T + 1e-8)))


            h_final = h_seq_v.numpy()[:, -1, :]
            pop_mean = np.mean(h_final, axis=1, keepdims=True)
            correlations = [
                np.corrcoef(h_final[:, i], pop_mean[:, 0])[0, 1]
                for i in range(h_final.shape[1])
            ]
            interference = np.mean(np.abs(correlations))

            history["loss"].append(np.mean(epoch_loss))
            history["acc"].append(val_acc)
            history["hidden_metrics"].append({
                "effective_rank": eff_rank,
                "synchrony": sync_val,
                "entropy": entropy_val,
                "a_corr": acorr_val,
                "interference": interference
            })

            print(f"{epoch+1:<6} | {np.mean(epoch_loss):<7.3f} | {val_acc*100:<8.2f} | "
                  f"{eff_rank:<6.2f} | {sync_val:<6.3f} | {entropy_val:<6.2f} | {acorr_val:<7.3f}{status_msg}")
    
    # 2. SAVE LATEST WEIGHTS (Post-training)
    model.save_weights(f"latest_{name}_{SESSION_ID}.weights.h5")
    print(f"--- Finished {name}: Best and Latest weights saved. ---")
            
    return history

# --- 5. SAVE ---
# This map will hold everything: configs, histories, and final scores
master_results = {
    "config": {
        "hidden": HIDDEN,
        "base_strength": BASE_STRENGTH,
        "jitter": JITTER_SCALE,
        "period": PERIOD,
        "epochs": EPOCHS,
        "data_pct": DATA_PERCENT
    },
    "runs": {}
}

def safe_evaluate(model, dataset):
    """Evaluates accuracy in batches to prevent GPU OOM errors."""
    accs = []
    for x_batch, y_batch in dataset:
        logits, _ = model(x_batch, training=False)
        preds = np.argmax(logits.numpy(), axis=-1)
        accs.append(np.mean(preds == y_batch.numpy()))
    return np.mean(accs)


# --- 6. EXECUTION ---
def reset_env():
    tf.keras.backend.clear_session()
    random.seed(42); np.random.seed(42); tf.random.set_seed(42)
    gc.collect()

# SETUP DATA
num_train = int(len(x_train) * DATA_PERCENT)
num_val = int(len(x_val)*DATA_PERCENT)
num_test  = int(len(x_test) * DATA_PERCENT)

train_ds = tf.data.Dataset.from_tensor_slices((x_train[:num_train], y_train[:num_train])) \
    .shuffle(5000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val[:1000], y_val[:1000])) \
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds_subset = tf.data.Dataset.from_tensor_slices((x_test[:num_test], y_test[:num_test])) \
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Test Run Ready: Samples -> Train: {num_train}, Val: {num_val}, Data %: {DATA_PERCENT*100}")

# --- RUN 1: ACTIVE ---
# --- 5. EXECUTION & UNIQUE SAVING ---

master_results = {
    "session_id": SESSION_ID,
    "config": {
        "hidden": HIDDEN, "base_strength": BASE_STRENGTH,
        "jitter": JITTER_SCALE, "period": PERIOD, "epochs": EPOCHS
    },
    "runs": {}
}

def save_metrics_to_files(history, model, model_name, test_acc):
    cell = model.rnn1.cell
    total_params = model.count_params()
    
    # 1. GENERATE THE TEXT TABLE 
    log_lines = []
    log_lines.append(f"\n DATA LOG: {model_name}")
    log_lines.append(f" CONFIG: Hidden: {cell.units} | Params: {total_params:,} | F-Wgt: {cell.strength:.4f} | "
                     f"L-Slow (τ): {cell.lambda_slow:.3f} | Jitter: {JITTER_SCALE:.2f} | Period: {cell.period:.1f} | Data: {DATA_PERCENT*100:.2f}%")
    log_lines.append("="*145)
    header = f"{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6} | {'F-Wgt':<6}"
    log_lines.append(header)
    log_lines.append("-" * 145)

    for i in range(len(history['loss'])):
        m = history['hidden_metrics'][i]
        line = (f"{i+1:<6} | {history['loss'][i]:<7.3f} | {history['acc'][i]*100:<8.2f} | "
                f"{m['effective_rank']:<6.2f} | {m['synchrony']:<6.3f} | {m['entropy']:<6.2f} | "
                f"{m['a_corr']:<7.3f} | {m['interference']:<6.3f} | {cell.strength:<6.3f}")
        log_lines.append(line)
    
    log_lines.append("-" * 145)
    test_str = f"{test_acc*100:.2f}%" if test_acc is not None else "N/A"
    log_lines.append(f" FINAL PERFORMANCE: Validation Acc: {history['acc'][-1]*100:.2f}% | TEST ACCURACY: {test_str}")
    log_lines.append("="*145 + "\n")

    # Save to TXT (Appends so you don't lose previous runs)
    with open(f"log_{SESSION_ID}.txt", "a") as f:
        f.write("\n".join(log_lines))
    
    # Print to console as usual
    print("\n".join(log_lines))

    # 2. SAVE RAW DATA TO JSON
    json_data = {
        "session_id": SESSION_ID,
        "model_name": model_name,
        "config": {
            "hidden": cell.units, "strength": cell.strength, "tau": cell.lambda_slow,
            "jitter": JITTER_SCALE, "period": PERIOD, "data_pct": DATA_PERCENT
        },
        "history": history,
        "test_accuracy": float(test_acc)
    }
    
    # Simple conversion for numpy types
    def clean_types(obj):
        if isinstance(obj, dict): return {k: clean_types(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean_types(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64)): return float(obj)
        if isinstance(obj, (np.int32, np.int64)): return int(obj)
        return obj

    with open(f"results_{SESSION_ID}_{model_name}.json", "w") as f:
        json.dump(clean_types(json_data), f, indent=4)



## Confusion matrix, salience map and symmetry calculation

In [ ]:
from sklearn.metrics import confusion_matrix

def plot_resonator_confusion(model, x_test_full, y_test_full, data_percent=DATA_PERCENT, reversed_mode=False, session_name="Session"):
    # 1. Respect the data constraint
    num_test = int(len(x_test_full) * data_percent)
    x_subset = x_test_full[:num_test]
    y_true = y_test_full[:num_test]

    # 2. Handle Reverse Mode (Internal labeling only)
    title_prefix = "REVERSED" if reversed_mode else "STANDARD"
    if reversed_mode:
        x_subset = x_subset[:, ::-1, :]

    # 3. Get predictions
    print(f"--> Predicting for {title_prefix}...")
    logits, _ = model.predict(x_subset, batch_size=BATCH_SIZE, verbose=0)
    y_pred = np.argmax(logits, axis=-1)
    y_true_np = y_true.numpy() if hasattr(y_true, 'numpy') else y_true

    # 4. Build ASCII Confusion Matrix
    cm = confusion_matrix(y_true_np, y_pred, labels=range(10))
    
    cm_lines = []
    cm_lines.append(f"\n{'#'*70}")
    cm_lines.append(f" {title_prefix} EVALUATION | SESSION: {session_name}")
    cm_lines.append(f"{'#'*70}")
    cm_lines.append("Pred:    0    1    2    3    4    5    6    7    8    9")
    cm_lines.append("True |" + "----" * 12)
    
    for i, row in enumerate(cm):
        row_str = f"{i:>3}  | " + " ".join([f"{val:>4}" for val in row])
        cm_lines.append(row_str)
    
    cm_lines.append("-" * 60)
    
    # 5. Calculate Bias Score
    most_common_idx = np.argmax(np.sum(cm, axis=0))
    bias_pct = (np.sum(cm[:, most_common_idx]) / np.sum(cm)) * 100
    cm_lines.append(f"Primary Bias: Digit '{most_common_idx}' accounts for {bias_pct:.1f}% of all predictions.")
    cm_lines.append(f"{'#'*70}\n")
    
    ascii_matrix = "\n".join(cm_lines)

    # 6. APPEND to the main session log ONLY (stripping any stray prefixes)
    # This ensures it goes into log_RES_H16...txt even if called weirdly
    clean_session = session_name.replace("Standard_", "").replace("Reversed_", "")
    log_filename = f"log_{clean_session}.txt"
    
    with open(log_filename, "a") as f:
        f.write(ascii_matrix)
    
    # Still print to console for the live check
    print(ascii_matrix)

In [ ]:
def plot_field_jitter(model, sample_batch):
    # 1. Extract hidden sequences
    logits, h_seq = model(sample_batch[:1], training=False)
    h_np = h_seq[0].numpy() # Shape: (784, HIDDEN)
    
    # 2. Reconstruct physics components
    base_delta = (2.0 * np.pi) / PERIOD
    steps = np.arange(h_np.shape[0])
    phase = steps * base_delta
    
    # Raw Oscillator and Neuronal Bias
    ghost_field = np.sin(phase)
    activity_bias = JITTER_SCALE * np.mean(h_np, axis=-1) 
    combined_signal = ghost_field + activity_bias
    
    # Reconstruct G_norm (Simplified EMA for visualization)
    # This approximates the 'new_G' EMA in your cell logic
    ema_g = 0
    g_history = []
    for step_h in h_np:
        # Shuffling the signal as per your 'half' concat logic
        half = len(step_h) // 2
        shuffled = np.concatenate([step_h[half:], step_h[:half]])
        ema_g = (1.0 - LAMBDA_SLOW) * ema_g + LAMBDA_SLOW * shuffled
        g_history.append(ema_g)
    
    g_history = np.array(g_history)
    # Z-score normalization for the tanh gating
    g_norm = (g_history - np.mean(g_history)) / (np.std(g_history) + 1e-6)
    
    # Calculate Final Gain Modulation (Field Effect)
    # FE = 1 + (strength * combined * tanh(G_norm))
    # We take the mean across neurons for the final visualization
    field_effect = 1.0 + (BASE_STRENGTH * combined_signal[:, None] * np.tanh(g_norm))
    field_effect_mean = np.mean(field_effect, axis=-1)

    # 3. Plotting
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    # Plot 1: Interference Pattern
    axes[0].plot(ghost_field, label="Pure Sine (Ghost)", color='gray', alpha=0.4, linestyle='--')
    axes[0].plot(combined_signal, label="Combined (Active)", color='#3498db', linewidth=1.5)
    axes[0].set_title("Carrier vs. Signal Interference")
    axes[0].legend()

    # Plot 2: Neuronal Bias (The DC Offset)
    axes[1].axhline(0, color='black', lw=1, alpha=0.3)
    axes[1].fill_between(steps, activity_bias, color='#e74c3c', alpha=0.3, label="Neuronal Bias")
    axes[1].plot(activity_bias, color='#e74c3c', lw=2)
    axes[1].set_title(f"Hidden State Pressure (Scale: {JITTER_SCALE})")
    axes[1].legend()

    # Plot 3: Resulting Gain Modulation (The Field Effect)
    axes[2].axhline(1.0, color='black', lw=1, alpha=0.3) # 1.0 is neutral gain
    axes[2].plot(field_effect_mean, color='#2ecc71', lw=2, label="Field Effect (Gain)")
    axes[2].fill_between(steps, 1.0, field_effect_mean, color='#2ecc71', alpha=0.2)
    axes[2].set_title("Total Gain Modulation ($FE_t$)")
    axes[2].set_ylabel("Multiplicative Factor")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

# Run it!
sample_x, sample_y = next(iter(test_ds_subset))

In [ ]:
def visualize_pmnist_complete(model, image, label, permutation):
    # 1. Setup Inverse Permutation
    inverse_perm = np.argsort(permutation)
    img_tensor = tf.convert_to_tensor(image[np.newaxis, ...], dtype=tf.float32)
    
    # 2. Manual hidden state extraction
    h1_seq = model.rnn1(img_tensor, training=False)
    h2_seq = model.rnn2(h1_seq, training=False)
    h3_seq = model.rnn3(h2_seq, training=False)
    
    # 3. Get Prediction and Gradients for Saliency
    with tf.GradientTape() as tape:
        tape.watch(img_tensor)        
        output = model(img_tensor, training=False)
        logits = output[0] if isinstance(output, tuple) else output
        
        # Calculate Prediction and Confidence
        probs = tf.nn.softmax(logits, axis=-1)
        pred_label = np.argmax(probs.numpy(), axis=-1)[0]
        confidence = np.max(probs.numpy(), axis=-1)[0]
        
        loss = logits[0, label]

    grads = tape.gradient(loss, img_tensor)
    saliency_flat = tf.reduce_max(tf.abs(grads), axis=-1).numpy().flatten()
    
    # 4. Prepare Images (Clean vs Scrambled)
    unscrambled_img = image.flatten()[inverse_perm].reshape(28, 28)
    unscrambled_sal = saliency_flat[inverse_perm].reshape(28, 28)
    scrambled_img = image.reshape(28, 28)
    scrambled_sal = saliency_flat.reshape(28, 28)

    # --- PLOTTING ---
    fig = plt.figure(figsize=(22, 12))
    
    # Color the title based on correctness
    result_color = "green" if pred_label == label else "red"
    
    # COLUMN 1: UNSCRAMBLED (The "Truth")
    ax1 = plt.subplot2grid((3, 5), (0, 0))
    ax1.imshow(unscrambled_img, cmap='gray')
    ax1.set_title(f"Target: {label} | PRED: {pred_label}\n({confidence*100:.1f}% Conf)", 
                  fontsize=14, color=result_color, fontweight='bold')
    
    ax2 = plt.subplot2grid((3, 5), (1, 0))
    ax2.imshow(unscrambled_sal, cmap='hot')
    ax2.set_title("Focus (Unscrambled)")

    # COLUMN 2: SCRAMBLED (The "Reality")
    ax3 = plt.subplot2grid((3, 5), (0, 1))
    ax3.imshow(scrambled_img, cmap='gray')
    ax3.set_title("Input (Scrambled)")
    
    ax4 = plt.subplot2grid((3, 5), (1, 1))
    ax4.imshow(scrambled_sal, cmap='hot')
    ax4.set_title("Focus (Scrambled)")

    # COLUMN 3-5: THE HIDDEN HEARTBEATS
    layers = [h1_seq, h2_seq, h3_seq]
    layer_names = ["Layer 1", "Layer 2", "Layer 3: Decision State"]
    
    for i, (h_seq, name) in enumerate(zip(layers, layer_names)):
        ax = plt.subplot2grid((3, 5), (i, 2), colspan=3)
        activity = tf.transpose(h_seq[0]).numpy()
        im = ax.imshow(activity, aspect='auto', cmap='magma', interpolation='nearest')
        ax.set_title(name)
        ax.set_ylabel("Neuron ID")
        if i == 2: ax.set_xlabel("Time (784 Steps)")
        plt.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.show()

# Run it!
#visualize_pmnist_complete(model_active, x_test[101], y_test[101], perm)

In [ ]:
from scipy.spatial.distance import euclidean
from scipy.stats import pearsonr

def correlate_jitter_fields(model, image_idx, x_data, period, strength, lambda_slow):
    # 1. Get Forward Jitter
    sample_fwd = x_data[image_idx:image_idx+1]
    _, h_fwd = model(sample_fwd, training=False)
    fe_fwd = calculate_field_effect(h_fwd[0].numpy(), period, strength, lambda_slow)

    # 2. Get Reversed Jitter
    sample_rev = sample_fwd[:, ::-1, :]
    _, h_rev = model(sample_rev, training=False)
    fe_rev = calculate_field_effect(h_rev[0].numpy(), period, strength, lambda_slow)
    
    # 3. Flip the Reversed signal back to compare 1:1 with Forward
    fe_rev_flipped = fe_rev[::-1]

    # 4. Math Metrics
    corr, _ = pearsonr(fe_fwd, fe_rev_flipped)
    dist = euclidean(fe_fwd, fe_rev_flipped)
    
    print(f"\n--- SYMMETRY ANALYSIS (Hash {image_idx}) ---")
    print(f"Correlation Score (R): {corr:.4f}  (1.0 = Perfect Symmetry)")
    print(f"Euclidean Distance:    {dist:.4f}  (0.0 = Identical Field)")
    
    return corr, dist

# Helper to match your plot_field_jitter logic
def calculate_field_effect(h_np, period, strength, lambda_slow):
    steps = np.arange(len(h_np))
    ghost = np.sin(steps * (2.0 * np.pi / period))
    bias = np.mean(h_np, axis=-1)
    combined = ghost + bias
    
    # Simple EMA to simulate G_norm
    ema_g = 0
    g_history = []
    for step_h in h_np:
        half = len(step_h) // 2
        shuffled = np.concatenate([step_h[half:], step_h[:half]])
        ema_g = (1.0 - lambda_slow) * ema_g + lambda_slow * shuffled
        g_history.append(ema_g)
    
    g_norm = (np.array(g_history) - np.mean(g_history)) / (np.std(g_history) + 1e-6)
    # Mean Field Effect across neurons
    fe = 1.0 + (strength * combined[:, None] * np.tanh(g_norm))
    return np.mean(fe, axis=-1)

from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

def analyze_symmetry(model, hash_std, hash_rev, x_data):
    # 1. Generate Fields
    def get_fe(idx, reversed=False):
        batch = x_data[idx:idx+1]
        if reversed: batch = batch[:, ::-1, :]
        _, h_seq = model(batch, training=False)
        return calculate_field_effect(h_seq[0].numpy(), PERIOD, BASE_STRENGTH, LAMBDA_SLOW)

    fe_std = get_fe(hash_std, reversed=False)
    fe_rev = get_fe(hash_rev, reversed=True)
    
    # Flip the reversed field to align 1:1 with the standard timeline
    fe_rev_aligned = fe_rev[::-1]

    # 2. Math Metrics
    corr, _ = pearsonr(fe_std, fe_rev_aligned)
    dist = euclidean(fe_std, fe_rev_aligned)

    print(f"\n{'='*40}")
    print(f" FIELD SYMMETRY: Hash {hash_std} vs Hash {hash_rev}")
    print(f"{'='*40}")
    print(f" Pearson Correlation: {corr:.4f}  (Target: 1.0)")
    print(f" Euclidean Distance:  {dist:.4f}  (Target: 0.0)")
    print(f"{'='*40}\n")
    
    # Run your existing plots
    plot_field_jitter(model, x_data[hash_std:hash_std+1])
    plot_field_jitter(model, x_data[hash_rev:hash_rev+1][:, ::-1, :])





## Execute

In [ ]:
# --- RUN 1: ACTIVE ---

print(f"\n[Phase 1] Training Active for {SESSION_ID}...")

model_active = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode="active")
_ = model_active(tf.zeros((1, 784, 1))) 

# We pass the SESSION_ID into the name so the .h5 file is unique
hist_active = train_phase(model_active, train_ds, val_ds, epochs=EPOCHS, name=f"{SESSION_ID}_active")

test_acc_active = safe_evaluate(model_active, test_ds_subset)

# Store everything in the map using the unique weight filename
master_results["runs"]["active"] = {
    "history": hist_active,
    "test_acc": float(test_acc_active),
    "weight_path": f"best_{SESSION_ID}_active.weights.h5"
}

# --- FINAL GLOBAL SAVE ---

print(f"\n SESSION COMPLETE!")
print(f"Weights: best_{SESSION_ID}_active.weights.h5")
save_metrics_to_files(hist_active, model_active, "Active", test_acc_active)

 
# --- RUN 2: PROBE ---

reset_env()
print(f"\n[Phase 2] Training Probe for {SESSION_ID}...")

model_probe = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode="probe")
_ = model_probe(tf.zeros((1, 784, 1))) 

# Train with unique ID
hist_probe = train_phase(model_probe, train_ds, val_ds, epochs=EPOCHS, name=f"{SESSION_ID}_probe")

test_acc_probe = safe_evaluate(model_probe, test_ds_subset)

# Store in master map
master_results["runs"]["probe"] = {
    "history": hist_probe,
    "test_acc": float(test_acc_probe),
    "weight_path": f"best_{SESSION_ID}_probe.weights.h5"
}

# Log to files
save_metrics_to_files(hist_probe, model_probe, "Probe", test_acc_probe)

# --- RUN 3: PASSIVE ---

reset_env()
print(f"\n[Phase 3] Training Passive for {SESSION_ID}...")

model_passive = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode="passive")
_ = model_passive(tf.zeros((1, 784, 1))) 

# Train with unique ID
hist_passive = train_phase(model_passive, train_ds, val_ds, epochs=EPOCHS, name=f"{SESSION_ID}_passive")

test_acc_passive = safe_evaluate(model_passive, test_ds_subset)

# Store in master map
master_results["runs"]["passive"] = {
    "history": hist_passive,
    "test_acc": float(test_acc_passive),
    "weight_path": f"best_{SESSION_ID}_passive.weights.h5"
}

# Log to files
save_metrics_to_files(hist_passive, model_passive, "Passive", test_acc_passive)


## Evaluate Confusion matrix

In [ ]:
# 1. Configuration for this eval run
DATA_PERCENT = 0.05
BATCH_SIZE = 128 
SESSION_ID = 'RES_H16_S0.22_J0.65_260324_1110'
# 2. Dynamic weight pathing using the SESSION_ID
# This ensures we aren't accidentally evaluating the wrong model version
# --- EVALUATION BLOCK (Mode-Agnostic Version) ---

# Set this once at the top of your cell depending on which you are testing
CURRENT_MODE = "active"  # or "probe" or "passive"

# 1. Dynamic pathing (No elif needed!)
best_weight_path = f"best_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5" 

print(f"--> Target Mode: {CURRENT_MODE}")
print(f"--> Weight Path: {best_weight_path}")

# 2. Universal Instantiation
if f'model_{CURRENT_MODE}' not in locals():
    print(f"--> Creating model_{CURRENT_MODE}...")
    # This works for any mode!
    model_to_eval = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode=CURRENT_MODE)
    _ = model_to_eval(tf.zeros((1, 784, 1))) 
else:
    model_to_eval = locals()[f'model_{CURRENT_MODE}']

# 3. Universal Load
if os.path.exists(best_weight_path):
    model_to_eval.load_weights(best_weight_path)
    print(f"--> [SUCCESS] Weights loaded for {CURRENT_MODE}")
else:
    print(f"--> [ERROR] Weights NOT FOUND at {best_weight_path}")


if 'model_active' not in locals():
    print(f"--> model_active not found. Re-instantiating for {SESSION_ID}...")
    model_active = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode="active")
    _ = model_active(tf.zeros((1, 784, 1))) 

if os.path.exists(best_weight_path):
    print(f"--> Restoring weights from: {best_weight_path}")
    model_active.load_weights(best_weight_path)
else:
    if os.path.exists("best_active.weights.h5"):
        print("--> Using generic best_active.weights.h5")
        model_active.load_weights("best_active.weights.h5")
    else:
        print(f"--> CRITICAL: No weights found for {SESSION_ID}!")


In [ ]:

# 3. Standard Evaluation
print("\nEvaluating Standard Test Set...")
num_test = int(len(x_test) * DATA_PERCENT)
x_subset = x_test[:num_test]
y_subset = y_test[:num_test]

# Using predict() for OOM safety
logits_std, _ = model_active.predict(x_subset, batch_size=BATCH_SIZE, verbose=1)
y_pred_std = np.argmax(logits_std, axis=-1)
test_acc_active = np.mean(y_pred_std == y_subset)

# 4. Reversed Evaluation
print("\nEvaluating Reversed Test Set...")
x_rev = x_subset[:, ::-1, :]
logits_rev, _ = model_active.predict(x_rev, batch_size=BATCH_SIZE, verbose=1)
y_pred_rev = np.argmax(logits_rev, axis=-1)
rev_acc_active = np.mean(y_pred_rev == y_subset) * 100

# 5. Final Stats & Logging logic
retention_ratio = rev_acc_active / (test_acc_active * 100 + 1e-9)

# Format the summary string
summary_box = [
    "\n" + "="*60,
    f" SESSION: {SESSION_ID}_{CURRENT_MODE}",
    f" REVERSAL MEMORY CHECK (Data: {DATA_PERCENT*100:.1f}%)",
    "="*60,
    f" Standard pMNIST Acc : {test_acc_active*100:>6.2f}%",
    f" Reversed pMNIST Acc : {rev_acc_active:>6.2f}%",
    f" Retention Ratio     : {retention_ratio:>6.2f}x",
    "="*60 + "\n"
]
summary_text = "\n".join(summary_box)

# APPEND to the text file
log_filename = f"log_{SESSION_ID}.txt"
with open(log_filename, "a") as f:
    f.write(summary_text)

print(summary_text)

# 6. Confusion Matrices (Already appends to log internally)
plot_resonator_confusion(model_to_eval, x_test, y_test, 
                         data_percent=DATA_PERCENT, 
                         reversed_mode=False, 
                         session_name=f"Standard_{SESSION_ID}")

plot_resonator_confusion(model_to_eval, x_test, y_test, 
                         data_percent=DATA_PERCENT, 
                         reversed_mode=True, 
                         session_name=f"Reversed_{SESSION_ID}")



In [ ]:
import os
import tensorflow as tf
import numpy as np

# --- 1. CONFIGURATION ---
DATA_PERCENT = 0.1  
BATCH_SIZE = 64
SESSION_ID = 'RES_H16_S0.22_J0.65_260324_1110'
MODES_TO_EVAL = ["probe", "passive", "active"]
USE_LATEST = False  

# Define the exact log filename
log_filename = f"log_{SESSION_ID}.txt"

# Helper to print to console AND append to log file simultaneously
def log_print(msg):
    print(msg)
    with open(log_filename, "a") as f:
        f.write(str(msg) + "\n")

# --- 2. EVALUATION LOOP ---
for CURRENT_MODE in MODES_TO_EVAL:
    log_print("\n" + "="*70)
    log_print(f" INITIALIZING EVALUATION: {CURRENT_MODE.upper()}")
    log_print("="*70)

    # Pathing logic for 'latest' vs 'best'
    if USE_LATEST:
        weight_path = f"latest_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5"
    else:
        weight_path = f"best_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5"
        #best_RES_H16_S0.22_J0.65_260223_2220_probe_RES_H16_S0.22_J0.65_260223_2220.weights
        #best_probe_RES_H16_S0.22_J0.65_260223_2220.weights.h5

    log_print(f"--> Target Mode: {CURRENT_MODE}")
    log_print(f"--> Weight Path: {weight_path}")

    # 3. Model Setup
    tf.keras.backend.clear_session()
    model_to_eval = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode=CURRENT_MODE)
    _ = model_to_eval(tf.zeros((1, 784, 1))) 

    # 4. Load Weights
    if os.path.exists(weight_path):
        model_to_eval.load_weights(weight_path)
        log_print(f"--> [SUCCESS] Weights loaded for {CURRENT_MODE}")
    else:
        log_print(f"--> [ERROR] Weights NOT FOUND at {weight_path}. Skipping...")
        continue

    # 5. Standard Evaluation
    log_print(f"\nEvaluating Standard Test Set ({CURRENT_MODE})...")
    num_test = int(len(x_test) * DATA_PERCENT)
    x_subset = x_test[:num_test]
    y_subset = y_test[:num_test]

    logits_std, _ = model_to_eval.predict(x_subset, batch_size=BATCH_SIZE, verbose=1)
    y_pred_std = np.argmax(logits_std, axis=-1)
    test_acc_raw = np.mean(y_pred_std == y_subset)

    # 6. Reversed Evaluation
    log_print(f"\nEvaluating Reversed Test Set ({CURRENT_MODE})...")
    x_rev = x_subset[:, ::-1, :] 
    logits_rev, _ = model_to_eval.predict(x_rev, batch_size=BATCH_SIZE, verbose=1)
    y_pred_rev = np.argmax(logits_rev, axis=-1)
    rev_acc_pct = np.mean(y_pred_rev == y_subset) * 100

    # 7. Stats calculation
    retention_ratio = rev_acc_pct / (test_acc_raw * 100 + 1e-9)
    weight_type_label = "LATEST" if USE_LATEST else "BEST"

    summary_box = [
        "\n" + "="*60,
        f" SESSION        : {SESSION_ID}",
        f" MODE           : {CURRENT_MODE.upper()} ({weight_type_label})",
        f" EVAL DATA %    : {DATA_PERCENT*100:.1f}%",
        "="*60,
        f" Standard Acc   : {test_acc_raw*100:>6.2f}%",
        f" Reversed Acc   : {rev_acc_pct:>6.2f}%",
        f" Retention Ratio: {retention_ratio:>6.2f}x",
        "="*60 + "\n"
    ]
    
    # Log the summary box
    for line in summary_box:
        log_print(line)

    # 8. Confusion Matrices 
    # (Assuming these handle their own internal logging/plotting)
    plot_resonator_confusion(model_to_eval, x_test, y_test, 
                             data_percent=DATA_PERCENT, 
                             reversed_mode=False, 
                             session_name=f"Standard_{SESSION_ID}_{CURRENT_MODE}")

    plot_resonator_confusion(model_to_eval, x_test, y_test, 
                             data_percent=DATA_PERCENT, 
                             reversed_mode=True, 
                             session_name=f"Reversed_{SESSION_ID}_{CURRENT_MODE}")

## Salience map code

In [ ]:
# Expanded scan to find the first 5 unique indices (Hashes) for each digit
def get_sample_summary_expanded(y_data):
    summary = {}
    # Target order: 1, 2, 3, 4, 5, 6, 7, 8, 9, 0
    target_order = [1, 2, 3, 4, 5, 6, 7, 8, 9, 0]
    
    for digit in target_order:
        # Find all indices where y_test matches the digit
        # Using y_data.numpy() if it's a tensor to ensure compatibility
        y_vals = y_data.numpy() if hasattr(y_data, 'numpy') else y_data
        indices = np.where(y_vals == digit)[0]
        summary[digit] = indices[:5].tolist() # Take the first five
        
    print(f"{'='*65}")
    print(f"            EXPANDED RESONATOR HASH DIRECTORY (Top 5)")
    print(f"{'='*65}")
    print(f"Digit | Hash 1 | Hash 2 | Hash 3 | Hash 4 | Hash 5")
    print(f"------|--------|--------|--------|--------|--------")
    for digit in target_order:
        h = summary[digit]
        # Padding in case some digits have fewer than 5 samples in the subset
        h += ["N/A"] * (5 - len(h))
        print(f"  {digit}   | {h[0]:>6} | {h[1]:>6} | {h[2]:>6} | {h[3]:>6} | {h[4]:>6}")
    print(f"{'='*65}")

# Run this on your subset
get_sample_summary_expanded(y_subset)

In [ ]:
import os
import tensorflow as tf
import numpy as np

# --- 1. CONFIGURATION ---
DATA_PERCENT = 0.01  
BATCH_SIZE = 128
SESSION_ID = 'RES_H16_S0.22_J0.65_260324_1110'
MODES_TO_EVAL = ["active"]
USE_LATEST = True  

# Define the exact log filename
log_filename = f"log_{SESSION_ID}.txt"

# Helper to print to console AND append to log file simultaneously
def log_print(msg):
    print(msg)
    with open(log_filename, "a") as f:
        f.write(str(msg) + "\n")

# --- 2. EVALUATION LOOP ---
for CURRENT_MODE in MODES_TO_EVAL:
    log_print("\n" + "="*70)
    log_print(f" INITIALIZING EVALUATION: {CURRENT_MODE.upper()}")
    log_print("="*70)

    # Pathing logic for 'latest' vs 'best'
    if USE_LATEST:
        weight_path = f"latest_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5"
    else:
        weight_path = f"best_{SESSION_ID}_{CURRENT_MODE}_{SESSION_ID}.weights.h5"
        #best_RES_H16_S0.22_J0.65_260223_2220_probe_RES_H16_S0.22_J0.65_260223_2220.weights
        #best_probe_RES_H16_S0.22_J0.65_260223_2220.weights.h5

    log_print(f"--> Target Mode: {CURRENT_MODE}")
    log_print(f"--> Weight Path: {weight_path}")

    # 3. Model Setup
    tf.keras.backend.clear_session()
    model_to_eval = OscillatingResonator(hidden=HIDDEN, strength=BASE_STRENGTH, mode=CURRENT_MODE)
    _ = model_to_eval(tf.zeros((1, 784, 1))) 

    # 4. Load Weights
    if os.path.exists(weight_path):
        model_to_eval.load_weights(weight_path)
        log_print(f"--> [SUCCESS] Weights loaded for {CURRENT_MODE}")
    else:
        log_print(f"--> [ERROR] Weights NOT FOUND at {weight_path}. Skipping...")
        continue


In [ ]:
# Visualize normal permuted sequence

Hash_p = 5
sample_batch = x_test[Hash_p:Hash_p+1]
visualize_pmnist_complete(model_to_eval, x_test[Hash_p], y_test[Hash_p], perm)

# Visualize reversed permuted sequence
Hash_rev = 5
rev_batch = x_test[Hash_rev:Hash_rev+1]
sample_batch_rev = rev_batch[:, ::-1, :]
visualize_pmnist_complete(model_to_eval, x_rev[Hash_rev], y_test[Hash_rev], perm[::-1])

# Analyze symetry:
analyze_symmetry(model_to_eval, Hash_p, Hash_rev, x_test)


## SOBOL analysis

In [ ]:
import numpy as np
import tensorflow as tf
from SALib.sample import saltelli
from SALib.analyze import sobol
import matplotlib.pyplot as plt
import gc
from datetime import datetime
import json
DATA_PERCENT = 0.1
BATCH_SIZE = 4 * 64
EPOCHS = 12         
HIDDEN = 32 
REST_BASELINE = 1.0
LEARNING_RATE = 1e-3
#BASE_STRENGTH =0.6
#LAMBDA_SLOW = 0.04
#PERIOD= 784/4 
#JITTER_SCALE = 0.6
N_baseline = 64 # This is the number you pass to saltelli.sample

# --- 1.1 AUTO-GENERATED SESSION NAME ---
# Creates a name like: RES_H32_S0.40_J1.15_T123456
now = datetime.now()
readable_ts = now.strftime("%y%m%d_%H%M") 
SESSION_ID = f"RES_H{HIDDEN}_S{BASE_STRENGTH}_J{JITTER_SCALE}_{readable_ts}"
print(f" SESSION INITIALIZED: {SESSION_ID}")
print_param_report(HIDDEN)

sobol_problem = {
    'num_vars': 5,
    'names': ['LAMBDA_SLOW', 'H_INERTIA','BASE_STRENGTH', 'PERIOD', 'JITTER_SCALE'],
    'bounds': [
        [0.001, 0.05], 
        [0.01, 0.99],
        [0.01, 0.99],      
        [784/10, 784/2], 
        [0.1, 1.2]
    ]
}

def calc_sobol_samples(problem, N, second_order=True):
    """
    Calculates total runs for a SALib Saltelli/Sobol sequence.
    Formula: N * (2D + 2) if second_order=True
             N * (D + 2)  if second_order=False
    """
    D = problem['num_vars']
    if second_order:
        total = N * (2 * D + 2)
    else:
        total = N * (D + 2)
    return total

# Your specific setup

total_runs = calc_sobol_samples(sobol_problem, N_baseline, second_order=True)

print(f"--- SOBOL RUN ESTIMATION ---")
print(f"Variables (D): {sobol_problem['num_vars']}")
print(f"Baseline (N):  {N_baseline}")
print(f"Total Model Evaluations: {total_runs}")

# Time estimation (Optional but helpful for dissertations)
avg_time_per_run = EPOCHS*24 # seconds (Estimate based on your Epochs)
total_hours = (total_runs * avg_time_per_run) / 3600
total_days = total_hours/24
print(f"Estimated Time: {total_hours:.2f} hours")
print(f"Estimated Time: {total_days:.2f} days")


## Oscilatory 

In [ ]:
import tensorflow as tf
import numpy as np
import scipy.linalg
import json
import gc
import math
from tqdm.auto import tqdm

# --- 1. GLOBAL MASTER LOGGING ---
SOBOL_MASTER_DATA = {"session_id": SESSION_ID, "runs": {}, "Si_all": {}}

def update_json_log(history, model, run_id):
    """Saves every epoch's progress into one master 'Big Ass' JSON file."""
    global SOBOL_MASTER_DATA
    if run_id is None: return # Skip if not a Sobol run
    
    cell = model.rnn1.cell
    SOBOL_MASTER_DATA["runs"][str(run_id)] = {
        "config": {
            "tau": float(cell.lambda_slow),
            "strength": float(cell.strength),
            "jitter": float(JITTER_SCALE),
            "period": float(cell.period),
            #"learning rate": float(LEARNING_RATE),
            "h_inertia": float(H_SCALE[0])
        },
        "history": history
    }

    def clean(obj):
        if isinstance(obj, dict): return {k: clean(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64, np.ndarray)): 
            return float(obj) if np.isscalar(obj) else obj.tolist()
        return obj

    with open(f"SOBOL_MASTER_{SESSION_ID}.json", "w") as f:
        json.dump(clean(SOBOL_MASTER_DATA), f, indent=4)

# --- 2. THE JITTERED CELL & MODEL ---
class JitteredFeedbackCell(tf.keras.layers.Layer):
    def __init__(self, units=16, strength=0.0, period=256.0, lambda_slow=0.05, mode="active", **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = [units, units, 1] 
        self.strength = strength 
        self.period = period
        self.lambda_slow = lambda_slow
        self.mode = mode

    def build(self, input_shape):
        self.w_in = self.add_weight(shape=(input_shape[-1], self.units), name="w_in", initializer="glorot_uniform")
        self.w_rec = self.add_weight(shape=(self.units, self.units), name="w_rec",
                                     initializer=tf.keras.initializers.Orthogonal(gain=1.0))
        self.bias = self.add_weight(shape=(self.units,), name="bias", initializer="zeros")

    def call(self, inputs, states):
        prev_h, prev_G, prev_phase = states
        half = self.units // 2
        raw_signal = tf.concat([prev_h[:, half:], prev_h[:, :half]], axis=1)
        source_signal = tf.stop_gradient(raw_signal) if self.mode == "probe" else raw_signal
        
        new_G = (1.0 - self.lambda_slow) * prev_G + self.lambda_slow * source_signal
        G_norm = (new_G - tf.reduce_mean(new_G, axis=-1, keepdims=True)) / (tf.math.reduce_std(new_G, axis=-1, keepdims=True) + 1e-6)
        
        new_phase = prev_phase + (2.0 * math.pi / self.period)
        oscillator = tf.math.sin(new_phase)
        
        if self.mode == "active":
            bias_signal = tf.reduce_mean(source_signal, axis=-1, keepdims=True) - 0.1
            combined_signal = oscillator + (JITTER_SCALE * bias_signal)
        else:
            combined_signal = oscillator

        current_strength = self.strength * combined_signal if self.mode != "passive" else 0.0
        field_effect = REST_BASELINE + (current_strength * tf.tanh(G_norm))
        
        z = (tf.matmul(inputs, self.w_in) + tf.matmul(prev_h, self.w_rec) + self.bias) * field_effect
        h = (H_SCALE[0] * prev_h) + (H_SCALE[1] * tf.nn.elu(z))
        h = tf.clip_by_value(h, -20.0, 20.0)
        return h, [h, new_G, new_phase]

class OscillatingResonator(tf.keras.Model):
    def __init__(self, hidden=16, num_classes=10, strength=0.0, mode="active"):
        super().__init__()
        # Use cell_ref so we can access hyperparameters easily later
        self.cell_ref = JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode)
        self.rnn1 = tf.keras.layers.RNN(self.cell_ref, return_sequences=True)
        self.rnn2 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.rnn3 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.out = tf.keras.layers.Dense(num_classes)

    def call(self, x, training=False):
        h1 = self.rnn1(x, training=training)
        h2 = self.rnn2(h1, training=training)
        h3 = self.rnn3(h2, training=training)
        return self.out(h3[:, -1, :]), h3

# --- 3. UPDATED TRAINING PHASE ---
def train_phase(model, train_data, val_data, epochs=3, name="model", run_id=None):
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    history = {"loss": [], "acc": [], "hidden_metrics": []}
    best_val_acc = 0.0

    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            logits, _ = model(x, training=True)
            loss_v = loss_fn(y, logits)
        grads = tape.gradient(loss_v, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss_v, logits

    # Added 'Intf' to the console header
    print(f"\n{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6}")
    print("-" * 85)

    for epoch in range(epochs):
        epoch_losses = []
        acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
        pbar = tqdm(train_data, desc=f"EPOCH {epoch+1}/{epochs}", leave=False)
        
        for x_b, y_b in pbar:
            loss_v, logits = train_step(x_b, y_b)
            acc_metric.update_state(y_b, logits)
            epoch_losses.append(float(loss_v))
            pbar.set_postfix({"loss": f"{np.mean(epoch_losses):.4f}", "acc": f"{acc_metric.result():.2%}"})

        # --- VALIDATION SNAPSHOT ---
        for x_v, y_v in val_data.take(1):
            logits_v, h_seq_v = model(x_v, training=False)
            val_acc = np.mean(np.argmax(logits_v.numpy(), axis=-1) == y_v.numpy())            
            
            # --- METRIC CALCULATIONS ---
            h_final = h_seq_v.numpy()[:, -1, :]
            
            # 1. Effective Rank
            s = scipy.linalg.svdvals(h_final) + 1e-12
            p_rank = s / (np.sum(s) + 1e-10)
            eff_rank = np.exp(-np.sum(p_rank * np.log(p_rank + 1e-10)))
            
            # 2. Entropy
            counts, _ = np.histogram(h_final, bins=50)
            p_ent = counts / (h_final.size + 1e-10) 
            entropy_val = -np.sum(p_ent * np.log2(p_ent + 1e-10))
            
            # 3. Synchrony (Inter-neuron correlation)
            sync_val = (np.sum(np.abs(np.corrcoef(h_final.T + 1e-8))) - HIDDEN) / (HIDDEN**2 - HIDDEN)
            
            # 4. Auto-Correlation (Temporal slowness)
            acorr_val = np.mean(np.abs(np.corrcoef(h_seq_v.numpy()[0].T + 1e-8)))

            # 5. NEW: Interference (Neuron-to-Field alignment)
            # Measures how much neurons are driven by the common field vs unique input
            mean_field = np.mean(h_final, axis=1, keepdims=True)
            neuron_to_field_corrs = [
                np.abs(np.corrcoef(h_final[:, j], mean_field[:, 0])[0, 1]) 
                for j in range(h_final.shape[1])
            ]
            interference_val = np.mean(np.nan_to_num(neuron_to_field_corrs))

        # --- HISTORY UPDATE ---
        avg_loss = np.mean(epoch_losses)
        history["loss"].append(float(avg_loss))
        history["acc"].append(float(val_acc))
        history["hidden_metrics"].append({
            "effective_rank": float(eff_rank),
            "synchrony": float(sync_val),
            "entropy": float(entropy_val),
            "a_corr": float(acorr_val),
            "interference": float(interference_val)
        })

        # --- THE BIG JSON SAVE ---
        update_json_log(history, model, run_id)
        
        # Unified Console Output
        print(f"EP {epoch+1}/{epochs}: {avg_loss:<7.3f} | {val_acc:<8.2%} | {eff_rank:<6.2f} | "
              f"{sync_val:<6.3f} | {entropy_val:<6.2f} | {acorr_val:<7.3f} | {interference_val:<6.3f}")

    return history

## Execute SOBOL

In [ ]:
# --- 7. SOBOL SENSITIVITY ANALYSIS IMPLEMENTATION (FIXED) ---

SOBOL_MASTER_DATA = {
    "session_id": SESSION_ID,
    "problem": sobol_problem,
    "runs": {},
    "Si_all": {} 
}

def evaluate_sobol_run(params, run_id, total_runs):
    # Map back to the globals the model uses
    global LEARNING_RATE, JITTER_SCALE, BASE_STRENGTH, PERIOD, LAMBDA_SLOW, H_INERTIA, H_SCALE
    
    current_lslow, h_inertia, current_strength, current_period, current_jitter = params
    
    LAMBDA_SLOW = current_lslow
    H_INERTIA = h_inertia          
    BASE_STRENGTH = current_strength
    PERIOD = current_period
    JITTER_SCALE = current_jitter
    H_SCALE = [H_INERTIA, 1.0 - H_INERTIA] 

    tf.keras.backend.clear_session()
    gc.collect()
    
    tf.keras.utils.set_random_seed(42)
    model = OscillatingResonator(hidden=HIDDEN, num_classes=10, strength=BASE_STRENGTH, mode="active")
    
    # Print Run Header
    print(f"\r>>> [RUNNING {run_id+1}/{total_runs}] "
          f"H_Inertia: {H_INERTIA:.3f} | Tau: {LAMBDA_SLOW:.3f} | Jitter: {JITTER_SCALE:.2f}...", end="")

    # Execute training (Assuming train_phase prints its own progress)
    history = train_phase(model, train_ds, val_ds, epochs=EPOCHS, name="sobol", run_id=run_id)
    
    # Extract metrics
    metrics = {
        "acc": float(history['acc'][-1]),
        "rank": float(history['hidden_metrics'][-1]['effective_rank']),
        "sync": float(history['hidden_metrics'][-1]['synchrony']),
        "entr": float(history['hidden_metrics'][-1]['entropy']),
        "acorr": float(history['hidden_metrics'][-1]['a_corr']),
        "Intf": float(history['hidden_metrics'][-1]['interference'])
    }
    
    del model
    return metrics

def run_sobol_analysis():
    print(f"--- STARTING MULTI-METRIC SOBOL ANALYSIS ({SESSION_ID}) ---")
    
    # SALib generates N * (2D + 2) samples
    # For D=5, N=8, this results in 8 * (10 + 2) = 96 runs
    param_values = saltelli.sample(sobol_problem, N_baseline, calc_second_order=True) 
    num_runs = len(param_values)
    
    metric_keys = ["acc", "rank", "sync", "entr", "acorr", "Intf"]
    Y = {m: np.zeros(num_runs) for m in metric_keys}

    for i, params in enumerate(param_values):
        try:
            # Pass num_runs to the evaluator for the header
            res = evaluate_sobol_run(params, i, num_runs)
            
            for key in metric_keys: 
                Y[key][i] = res[key]
            
            # CLEAR PREVIOUS OUTPUT: \r returns cursor to start, \033[K clears to end of line
            # This replaces the bulky training logs with a single clean summary line
            print(f"\r\033[K✓ Run {i+1}/{num_runs} Complete | Acc: {res['acc']:.4f} | Sync: {res['sync']:.3f}", end="")
            
        except Exception as e:
            print(f"\n!! Run {i} failed: {e}")
            for key in metric_keys: Y[key][i] = np.nan 

    print("\n\n--- CALCULATING SENSITIVITY INDICES ---")
    # Calculate Sensitivity Indices for all tracked metrics
    Si_all = {}
    for key in metric_keys:
        y_clean = Y[key]
        nan_mask = np.isnan(y_clean)
        n_failed = np.sum(nan_mask)
        
        if n_failed > 0:
            print(f"Warning: {n_failed} failed runs for metric '{key}'")
        
        # If more than 10% of runs failed, the indices might be unreliable
        if n_failed / len(y_clean) > 0.1:
            print(f"Skipping {key}: failure rate too high ({n_failed}/{len(y_clean)})")
            Si_all[key] = None
            continue
            
        # Perform Sobol Analysis
        Si = sobol.analyze(sobol_problem, y_clean, print_to_console=False)
        Si_all[key] = {
            "S1": Si['S1'].tolist(),
            "ST": Si['ST'].tolist(),
            "S2": Si['S2'].tolist() if 'S2' in Si else None,
            "n_failed": int(n_failed)
        }

    SOBOL_MASTER_DATA["Si_all"] = Si_all
    
    # JSON Serialization helper for Numpy/Tensorflow types
    def clean(obj):
        if isinstance(obj, dict): return {k: clean(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64, np.ndarray)): 
            return float(obj) if np.isscalar(obj) else obj.tolist()
        return obj

    output_path = f"SOBOL_MASTER_{SESSION_ID}.json"
    with open(output_path, "w") as f:
        json.dump(clean(SOBOL_MASTER_DATA), f, indent=4)
        
    print(f"\nFINISH. Full sensitivity results saved to: {output_path}")
    return Si_all

# --- EXECUTION ---
Si_all = run_sobol_analysis()

## Evaluate SOBOL

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. LOAD THE DATA ---
FILE_PATH = r"SOBOL_RECOVERED_S2.json"

with open(FILE_PATH, "r") as f:
    data = json.load(f)

# --- 2. HEATMAP: SENSITIVITY INDICES ---
def plot_sensitivity_heatmap(data):
    si_all = data.get("Si_all", {})
    param_names = data["problem"]["names"]
    
    # Restructure data for Seaborn
    heatmap_data = []
    metrics = list(si_all.keys()) # ['acc', 'rank', 'sync', 'entr', 'acorr', 'Intf']
    
    for metric in metrics:
        # We focus on ST (Total-Order Sensitivity) for the big picture
        st_values = si_all[metric]["ST"]
        heatmap_data.append(st_values)
        
    df = pd.DataFrame(heatmap_data, index=metrics, columns=param_names)
    
    plt.figure(figsize=(10, 6))
    sns.heatmap(df, annot=True, cmap="viridis", fmt=".2f", linewidths=0.5)
    plt.title("Total-Order Sensitivity ($S_T$) Across All Metrics", pad=15)
    plt.ylabel("Measurement Metric")
    plt.xlabel("Hyperparameter")
    plt.tight_layout()
    plt.show()

# --- 3. EVOLUTION PLOTS: EPOCH TRAJECTORIES ---
def plot_epoch_evolution(data):
    runs = data.get("runs", {})
    
    # Setup a 2x3 grid for the 6 measurements
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Epoch-by-Epoch Evolution Across All Sobol Runs", fontsize=16)
    axes = axes.flatten()
    
    metrics_to_plot = [
        ("acc", "Accuracy"),
        ("effective_rank", "Effective Rank"),
        ("synchrony", "Synchrony"),
        ("entropy", "Entropy"),
        ("a_corr", "Auto-Correlation"),
        ("interference", "Interference")
    ]
    
    for ax, (metric_key, title) in zip(axes, metrics_to_plot):
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Value")
        ax.grid(True, linestyle="--", alpha=0.6)
        
        # Plot every run as a semi-transparent line
        for run_id, run_data in runs.items():
            try:
                if metric_key == "acc":
                    y_vals = run_data["history"]["acc"]
                else:
                    y_vals = [epoch_data[metric_key] for epoch_data in run_data["history"]["hidden_metrics"]]
                
                x_vals = range(1, len(y_vals) + 1)
                # alpha=0.15 creates a "density" effect when hundreds of lines overlap
                ax.plot(x_vals, y_vals, color="blue", alpha=0.15, linewidth=1)
            except KeyError:
                continue # Skip if a run crashed or is missing data

    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

# --- EXECUTE ---
if __name__ == "__main__":
    plot_sensitivity_heatmap(data)
    plot_epoch_evolution(data)

In [ ]:
import json
import numpy as np
import os
from SALib.analyze import sobol

# --- CONFIGURATION ---
FILE_NAME = "SOBOL_RECOVERED_S2.json"
METRIC_KEYS = ["acc", "rank", "sync", "entr", "acorr", "Intf"]

# --- STEP 1: LOAD DATA ---
if not os.path.exists(FILE_NAME):
    print(f"Error: {FILE_NAME} not found.")
    exit()

with open(FILE_NAME, "r") as f:
    master_data = json.load(f)

problem = master_data["problem"]
runs = master_data["runs"]

# Filter for runs that actually have history data
valid_ids = sorted([k for k in runs.keys() if "history" in runs[k]], key=int)
num_runs = len(valid_ids)

print(f"Loaded {num_runs} valid runs. Processing metrics...")

# --- STEP 2: EXTRACT Y-VALUES ---
# We map your JSON keys to the internal Y_data keys
Y_data = {m: np.zeros(num_runs) for m in METRIC_KEYS}

for i, run_id in enumerate(valid_ids):
    history = runs[run_id]["history"]
    
    # Extracting from the LAST entry of the lists (Final Epoch)
    # acc is a direct list in history
    Y_data["acc"][i] = history["acc"][-1]
    
    # hidden_metrics is a list of dicts; we take the last dict
    last_metrics = history["hidden_metrics"][-1]
    
    Y_data["rank"][i]  = last_metrics["effective_rank"]
    Y_data["sync"][i]  = last_metrics["synchrony"]
    Y_data["entr"][i]  = last_metrics["entropy"]
    Y_data["acorr"][i] = last_metrics["a_corr"]
    Y_data["Intf"][i]  = last_metrics["interference"]

# --- STEP 3: SOBOL ANALYSIS ---
Si_results = {}

for key in METRIC_KEYS:
    print(f"Calculating Sobol (S1, ST, S2) for: {key}...")
    
    # calc_second_order=True requires the specific sample size N*(2D+2)
    # SALib will throw an error if your num_runs doesn't match the problem definition
    try:
        Si = sobol.analyze(problem, Y_data[key], calc_second_order=True, print_to_console=False)
        
        # S2 is a NxN matrix, we must convert it to a list for JSON
        Si_results[key] = {
            "S1": Si['S1'].tolist(),
            "ST": Si['ST'].tolist(),
            "S2": Si['S2'].tolist() if Si['S2'] is not None else None,
            "S1_conf": Si['S1_conf'].tolist(),
            "ST_conf": Si['ST_conf'].tolist()
        }
    except Exception as e:
        print(f"  [!] Failed to calculate S2 for {key}: {e}")

# --- STEP 4: SAVE ---
master_data["Si_all"] = Si_results
output_file = FILE_NAME.replace(".json", "_FIXED_S2.json")

with open(output_file, "w") as f:
    json.dump(master_data, f, indent=4)

print(f"\nSUCCESS: Results saved to {output_file}")

In [ ]:
import json
import numpy as np
import re
import os
from SALib.analyze import sobol

#FILE_NAME = "SOBOL_MASTER_RES_H32_S0.22_J0.6_260319_1405.json"
FINAL_OUTPUT = "SOBOL_RECOVERED_S2.json"

def load_with_nan_fix(filename):
    print(f"Loading and sanitizing {filename}...")
    if not os.path.exists(filename):
        raise FileNotFoundError(f"Could not find {filename}")
        
    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()

    # The Fix: Replace unquoted NaN with null so the JSON parser accepts it
    sanitized = re.sub(r':\s*NaN', ': null', content, flags=re.IGNORECASE)
    sanitized = re.sub(r':\s*nan', ': null', sanitized)
    
    try:
        return json.loads(sanitized)
    except json.JSONDecodeError:
        # If it still fails, the file might be mid-write. Seal it.
        print("JSON incomplete, applying emergency seal...")
        if not sanitized.strip().endswith('}'):
            sanitized = sanitized.strip().rstrip(',') + "}}}}"
        return json.loads(sanitized)

# --- EXECUTION ---
try:
    master_data = load_with_nan_fix(FILE_NAME)
except Exception as e:
    print(f"FATAL ERROR during load: {e}")
    exit()

runs = master_data["runs"]
problem = master_data["problem"]
param_names = problem['names']

# Sort IDs numerically to ensure Y matches the Sobol sequence
valid_ids = sorted(runs.keys(), key=int)
num_runs = len(valid_ids)

print(f"Processing {num_runs} runs...")

# Define targets
metric_map = {
    "acc": "acc",
    "rank": "effective_rank",
    "sync": "synchrony",
    "entr": "entropy",
    "acorr": "a_corr",
    "Intf": "interference"
}

Y_data = {k: np.zeros(num_runs) for k in metric_map.keys()}

for i, rid in enumerate(valid_ids):
    hist = runs[rid].get("history", {})
    
    # 1. Grab last Accuracy
    acc_list = hist.get("acc", [])
    Y_data["acc"][i] = acc_list[-1] if acc_list else 0
    
    # 2. Grab last epoch of Hidden Metrics
    metrics_list = hist.get("hidden_metrics", [])
    if metrics_list:
        last_epoch = metrics_list[-1]
        for short_key, json_key in metric_map.items():
            if short_key == "acc": continue
            
            val = last_epoch.get(json_key)
            # If val is None (from our 'null' fix) or NaN, default to 0
            if val is None or (isinstance(val, float) and np.isnan(val)):
                Y_data[short_key][i] = 0.0
            else:
                Y_data[short_key][i] = float(val)

# --- STEP 3: DYNAMIC TRIMMING & ANALYSIS ---
Si_results = {}

D = problem['num_vars']
calc_second_order = True 
# For 5 vars: (2*5 + 2) = 12 runs per block
step_size = (2 * D + 2) if calc_second_order else (D + 2)

for key in Y_data.keys():
    total_available = len(Y_data[key])
    num_complete_blocks = total_available // step_size
    valid_cutoff = num_complete_blocks * step_size
    
    if valid_cutoff == 0:
        print(f"Skipping {key}: Need at least {step_size} runs, but only have {total_available}.")
        continue

    # Trim to valid Sobol structure
    Y_trimmed = Y_data[key][:valid_cutoff]
    
    print(f"\nAnalyzing {key.upper()}: Using {valid_cutoff} runs...")

    try:
        Si = sobol.analyze(problem, Y_trimmed, calc_second_order=calc_second_order, print_to_console=False)
        
        # --- S2 Extraction & Significance Check ---
        s2_raw = Si['S2']
        significant_interactions = []
        
        # S2 is a DxD matrix where only the upper triangle is filled
        for i in range(D):
            for j in range(i + 1, D):
                val = s2_raw[i, j]
                if not np.isnan(val) and abs(val) > 0.02:  # Threshold for significance
                    significant_interactions.append(f"{param_names[i]} x {param_names[j]}: {val:.3f}")
        
        if significant_interactions:
            print(f"  -> Top Interactions: {', '.join(significant_interactions[:2])}")

        # Store results (clean NaNs for JSON safety)
        Si_results[key] = {
            "S1": np.nan_to_num(Si['S1'], nan=0.0).tolist(),
            "ST": np.nan_to_num(Si['ST'], nan=0.0).tolist(),
            "S2": np.nan_to_num(Si['S2'], nan=0.0).tolist()
        }
        
    except Exception as e:
        print(f"  -> Sobol failed for {key}: {e}")

# --- STEP 4: SAVE ---
master_data["Si_all"] = Si_results
with open(FINAL_OUTPUT, "w") as f:
    json.dump(master_data, f, indent=4)

print(f"\nSUCCESS! Recovery and Analysis complete.")
print(f"Saved results to: {FINAL_OUTPUT}")

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# 1. LOAD THE PROPER DATA
FILE_NAME = "SOBOL_RECOVERED_S2.json" 
with open(FILE_NAME, "r") as f:
    master = json.load(f)

si_all = master["Si_all"]
param_names = master["problem"]["names"]
metrics = list(si_all.keys())

# --- 2. PLOT 1: TOTAL ORDER (ST) VS FIRST ORDER (S1) ---
# This shows direct vs. interaction-based drivers
st_df = pd.DataFrame({m: si_all[m]["ST"] for m in metrics}, index=param_names).T
s1_df = pd.DataFrame({m: si_all[m]["S1"] for m in metrics}, index=param_names).T

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

sns.heatmap(s1_df, annot=True, cmap="RdBu_r", ax=axes[0], cbar_kws={'label': 'Index Value'})
axes[0].set_title("First-Order Sensitivity ($S_1$)\n(Direct Contribution)")

sns.heatmap(st_df, annot=True, cmap="RdBu_r", ax=axes[1], cbar_kws={'label': 'Index Value'})
axes[1].set_title("Total-Order Sensitivity ($S_T$)\n(Including Interactions)")

plt.tight_layout()
plt.show()

# --- 3. PLOT 2: S2 INTERACTION MATRIX (Corrected for Object Dtype) ---
target_metric = "acc" 

# Convert the list-of-lists directly to a DataFrame and force float conversion
# This handles the 'None' to 'NaN' transition correctly
s2_matrix = pd.DataFrame(
    si_all[target_metric]["S2"], 
    index=param_names, 
    columns=param_names
).astype(float) # <--- This is the magic line that fixes your TypeError

plt.figure(figsize=(10, 8))

# Mask the lower triangle and the diagonal for a cleaner look
mask = np.tril(np.ones_like(s2_matrix, dtype=bool))

sns.heatmap(
    s2_matrix, 
    mask=mask, 
    annot=True, 
    cmap="RdBu_r", 
    center=0, 
    fmt=".3f",
    cbar_kws={'label': 'Synergy Index'}
)

plt.title(f"Second-Order Synergy ($S_2$) for {target_metric.upper()}\n"
          f"Positive = Synergy | Negative = Competitive")
plt.show()

# --- 4. PLOT 3: THE INTERACTION NETWORK ---
# Visualizing the "binding" strength between parameters
# --- 4. PLOT 3: THE INTERACTION NETWORK (FIXED) ---
plt.figure(figsize=(8, 8))
G = nx.Graph()

# Iterate through the DataFrame we created in Step 3
for i, p1 in enumerate(param_names):
    for j, p2 in enumerate(param_names):
        if i >= j: continue # Only look at the upper triangle
        
        val = s2_matrix.iloc[i, j] # Use iloc to get values from the DF
        
        # We only plot significant interactions (> 0.02 or 0.05)
        if not np.isnan(val) and abs(val) > 0.02: 
            color = 'firebrick' if val > 0 else 'royalblue'
            G.add_edge(p1, p2, weight=abs(val), color=color)

# Draw the graph
pos = nx.spring_layout(G, k=1.5) # Spring layout often looks better than circular for networks
edges = G.edges()
colors = [G[u][v]['color'] for u, v in edges]
weights = [G[u][v]['weight'] * 40 for u, v in edges] # Scaled for visibility

nx.draw(G, pos, with_labels=True, node_size=3000, node_color="#f0f0f0", 
        edge_color=colors, width=weights, font_size=10, font_weight="bold")

plt.title(f"Parameter Interaction Network: {target_metric.upper()}\n"
          f"(Red = Synergy | Blue = Competition)")
plt.show()

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import os

# --- 1. CONFIGURATION & MAPPING ---
# Ensure this matches your local filename
FILE_NAME = "SOBOL_RECOVERED_S2.json" 

# Mapping the "Problem" names to the "Config" keys found in your JSON
CONFIG_MAP = {
    "LAMBDA_SLOW": "tau",
    "BASE_STRENGTH": "strength",
    "JITTER_SCALE": "jitter",
    "PERIOD": "period",
    "H_INERTIA": "h_inertia",
    "LEARNING_RATE": "learning rate"
}

# List of all measures to plot
METRICS = [
    ("acc", "Accuracy"),
    ("loss", "Loss"),
    ("effective_rank", "Effective Rank"),
    ("synchrony", "Synchrony"),
    ("entropy", "Entropy"),
    ("a_corr", "Auto-Correlation"),
    ("interference", "Interference")
]

def generate_sobol_evolution_plots(file_path):
    with open(file_path, "r") as f:
        data = json.load(f)

    problem = data["problem"]
    runs = data["runs"]
    param_names = problem["names"]
    param_bounds = problem["bounds"]

    # Iterate through every metric (e.g., Accuracy, then Rank, etc.)
    for metric_key, metric_label in METRICS:
        print(f"Generating evolution plots for: {metric_label}...")
        
        # Create a figure with 5 subplots (one for each hyperparameter)
        fig, axes = plt.subplots(1, 5, figsize=(25, 6), sharey=False)
        fig.suptitle(f"Drift of {metric_label} Across Parameter Space (Epoch 1 → 6)", 
                     fontsize=22, fontweight='bold', y=1.08)
        
        for i, p_name in enumerate(param_names):
            ax = axes[i]
            c_key = CONFIG_MAP.get(p_name)
            bounds = param_bounds[i]
            
            # Formatting the subplot
            ax.set_title(f"vs {p_name}", fontsize=14, pad=10)
            ax.set_ylabel(f"Parameter Value: {p_name}", fontsize=12)
            ax.set_xlabel(f"{metric_label} Value", fontsize=12)
            ax.set_ylim(bounds[0], bounds[1])
            ax.grid(True, linestyle="--", alpha=0.3)
            
            # Plot trajectories for every single run
            for run_id, run_data in runs.items():
                try:
                    if "history" not in run_data or "config" not in run_data:
                        continue
                    
                    # 1. Get the hyperparameter value (Constant for the run - Y-axis)
                    y_val = run_data["config"][c_key]
                    
                    # 2. Get the metric history (Changes per epoch - X-axis)
                    if metric_key in ["acc", "loss"]:
                        x_history = run_data["history"][metric_key]
                    else:
                        # Extract from the hidden_metrics list
                        x_history = [epoch[metric_key] for epoch in run_data["history"]["hidden_metrics"]]
                    
                    # 3. Draw the line (Blue = trajectory, Red Dot = Final State)
                    ax.plot(x_history, [y_val] * len(x_history), 
                            color='#2980b9', alpha=0.1, linewidth=1)
                    
                    # Mark the final epoch (Epoch 5)
                    ax.scatter(x_history[-1], y_val, color='#c0392b', s=8, alpha=0.2)
                    
                except (KeyError, IndexError, TypeError):
                    continue # Skip incomplete data points silently
        
        plt.tight_layout()
        # Save as high-res PNG
        plt.savefig(f"SOBOL_EVOLUTION_{metric_key.upper()}.png", dpi=150, bbox_inches='tight')
        plt.show()

if __name__ == "__main__":
    generate_sobol_evolution_plots(FILE_NAME)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import parallel_coordinates

# --- 1. DATA PROCESSING ---
FILE_PATH = "SOBOL_RECOVERED_S2.json"
with open(FILE_PATH, "r") as f:
    data = json.load(f)

def get_processed_df(data):
    runs = data["runs"]
    param_names = data["problem"]["names"]
    # Mapping JSON config keys to readable problem names
    config_map = {"LAMBDA_SLOW": "tau", "BASE_STRENGTH": "strength", 
                  "JITTER_SCALE": "jitter", "PERIOD": "period", "H_INERTIA": "h_inertia"}
    #config_map = {"LEARNING_RATE": "LR", "H_INERTIA": "h_inertia"}
    
    
    records = []
    for rid, rdata in runs.items():
        if "history" not in rdata: continue
        h = rdata["history"]
        m = h["hidden_metrics"][-1]
        
        row = {
            "acc": h["acc"][-1],
            "loss": h["loss"][-1],
            "rank": m["effective_rank"],
            "sync": m["synchrony"],
            "entr": m["entropy"],
            "acorr": m["a_corr"],
            "intf": m["interference"]
        }
        # Safely map parameters
        for p_name in param_names:
            c_key = config_map.get(p_name, p_name.lower())
            row[p_name] = rdata["config"].get(c_key, 0)
        records.append(row)
    return pd.DataFrame(records)

df = get_processed_df(data)


In [ ]:

# --- 2. PLOT 1: RADAR SENSITIVITY (FIXED) ---
def plot_radar(si_all, param_names, target_metrics=['acc', 'sync', 'rank', 'entr', 'acorr', 'Intf']):
    angles = np.linspace(0, 2 * np.pi, len(param_names), endpoint=False).tolist()
    angles += angles[:1] # Close the circle
    
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    
    for m_key in target_metrics:
        if m_key in si_all:
            # Convert list to numpy array to ensure it works, then handle the loop
            values = np.array(si_all[m_key]["ST"]).tolist()
            values += values[:1]
            ax.plot(angles, values, linewidth=2, label=m_key.upper())
            ax.fill(angles, values, alpha=0.1)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(param_names)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.title("Total Sensitivity ($S_T$) across Metrics", pad=20, fontsize=15)
    plt.show()

# --- 3. PLOT 2: PHASE PORTRAIT (Synchrony vs Accuracy) ---
plt.figure(figsize=(10, 6))
# Using 'sync' and 'acc' from our processed dataframe
scatter = plt.scatter(df['sync'], df['acc'], c=df['H_INERTIA'], cmap='viridis', alpha=0.6)
plt.colorbar(scatter, label='H_INERTIA Value')
plt.xlabel('Global Synchrony (Final State)')
plt.ylabel('Accuracy')
plt.title('Mechanism: Accuracy vs. Synchrony Phase Portrait')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()



# --- 4. PLOT 3: PARALLEL COORDINATES (The "Winning Recipe") ---
def plot_winning_pathways(df):
    plt.figure(figsize=(15, 8))
    
    # 1. Prepare and Normalize Data
    cols = ['LAMBDA_SLOW', 'BASE_STRENGTH', 'JITTER_SCALE', 'PERIOD', 'H_INERTIA', 'acc']
    df_plot = df[cols].copy()
    
    # Min-Max Normalization for plotting
    for col in cols:
        df_plot[col] = (df_plot[col] - df_plot[col].min()) / (df_plot[col].max() - df_plot[col].min())
    
    # 2. Split into groups based on original Accuracy
    q90_val = df['acc'].quantile(0.99)
    # Get indices for splitting
    top_indices = df[df['acc'] >= q90_val].index
    other_indices = df[df['acc'] < q90_val].index
    
    x = np.arange(len(cols))
    
    # 3. Plot "Others" first (Green, Higher Alpha)
    for idx in other_indices:
        plt.plot(x, df_plot.loc[idx], color="#710d0d", alpha=0.15, linewidth=0.5)
        
    # 4. Plot "Top 1%" (Red, Lower Alpha)
    for idx in top_indices:
        plt.plot(x, df_plot.loc[idx], color="#3ce73c", alpha=0.6, linewidth=2.0)

    # 5. Formatting
    plt.xticks(x, cols, rotation=15, fontsize=12)
    plt.title("Neural Pathway Analysis: Top 10% (Red Ghosting) vs Others (Solid Green)", fontsize=16, pad=20)
    plt.ylabel("Normalized Parameter Range [0-1]", fontsize=12)
    
    # Custom Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color="#710d0d", lw=2, alpha=0.15, label='Others '),
        Line2D([0], [0], color="#3ce742", lw=2, alpha=0.6, label='Top 1% ')
    ]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.grid(axis='x', linestyle='-', alpha=0.1)
    plt.grid(axis='y', linestyle='--', alpha=0.2)
    plt.tight_layout()
    plt.show()


# --- EXECUTE ALL ---
plot_radar(data["Si_all"], data["problem"]["names"])
plot_winning_pathways(df)

def plot_rank_pathways(df):
    plt.figure(figsize=(15, 8))
    
    # 1. Prepare and Normalize Data
    # We include 'rank' in the columns now to see its final value
    cols = ['LAMBDA_SLOW', 'BASE_STRENGTH', 'JITTER_SCALE', 'PERIOD', 'H_INERTIA', 'rank']
    df_plot = df[cols].copy()
    
    # Min-Max Normalization for visual alignment
    for col in cols:
        df_plot[col] = (df_plot[col] - df_plot[col].min()) / (df_plot[col].max() - df_plot[col].min())
    
    # 2. Split into groups based on Effective Rank
    # Quantile 0.99 targets the most complex high-dimensional models
    q99_rank = df['rank'].quantile(0.99)
    top_rank_indices = df[df['rank'] >= q99_rank].index
    other_indices = df[df['rank'] < q99_rank].index
    
    x = np.arange(len(cols))
    
    # 3. Plot "Others" (Deep Red/Brown background)
    for idx in other_indices:
        plt.plot(x, df_plot.loc[idx], color="#710d0d", alpha=0.15, linewidth=0.5)
        
    # 4. Plot "Top 1% Rank" (Bright Green highlight)
    # Higher alpha and thicker lines to show the 'complex' pathways
    for idx in top_rank_indices:
        plt.plot(x, df_plot.loc[idx], color="#3ce73c", alpha=0.6, linewidth=2.5)

    # 5. Formatting
    plt.xticks(x, cols, rotation=15, fontsize=12)
    plt.title("Dimensionality Analysis: Top 1% Effective Rank (Green) vs Baseline (Red)", fontsize=16, pad=20)
    plt.ylabel("Normalized Parameter Range [0-1]", fontsize=12)
    
    # Custom Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color="#710d0d", lw=1, alpha=0.3, label='Standard Rank Runs'),
        Line2D([0], [0], color="#3ce73c", lw=3, alpha=0.8, label='Max Dimensionality (Top 1%)')
    ]
    plt.legend(handles=legend_elements, loc='upper right')
    
    plt.grid(axis='x', linestyle='-', alpha=0.1)
    plt.grid(axis='y', linestyle='--', alpha=0.2)
    plt.tight_layout()
    plt.show()

# --- EXECUTE ---
plot_rank_pathways(df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

def plot_combined_pathways(df):
    plt.figure(figsize=(16, 9))
    
    # 1. Prepare and Normalize Data
    # We include both outcomes at the end of the axis
    cols = ['LAMBDA_SLOW', 'BASE_STRENGTH', 'JITTER_SCALE', 'PERIOD', 'H_INERTIA', 'acc', 'rank']
    #cols = ['LEARNING_RATE','H_INERTIA', 'acc', 'rank']
    df_plot = df[cols].copy()
    
    for col in cols:
        df_plot[col] = (df_plot[col] - df_plot[col].min()) / (df_plot[col].max() - df_plot[col].min())
    
    # 2. Define our "Elite" groups
    q99_acc = df['acc'].quantile(0.95)
    q99_rank = df['rank'].quantile(0.95)
    
    acc_top_idx = df[df['acc'] >= q99_acc].index
    rank_top_idx = df[df['rank'] >= q99_rank].index
    # Others are anything not in the top 1% of either
    other_idx = df.index.difference(acc_top_idx.union(rank_top_idx))
    
    x = np.arange(len(cols))
    
    # 3. Plot "Others" (Grey/Brown background - Very low alpha)
    for idx in other_idx:
        plt.plot(x, df_plot.loc[idx], color="#710d0d", alpha=0.15, linewidth=0.8)
        
    # 4. Plot "Top 1% Rank" (Blue highlight - Dimensionality)
    for idx in rank_top_idx:
        plt.plot(x, df_plot.loc[idx], color="#3498db", alpha=0.6, linewidth=2.5)
        
    # 5. Plot "Top 1% Accuracy" (Green highlight - Performance)
    for idx in acc_top_idx:
        plt.plot(x, df_plot.loc[idx], color="#2ecc71", alpha=0.6, linewidth=2.5)

    # 6. Formatting
    plt.xticks(x, cols, rotation=15, fontsize=12)
    plt.title("Dual-Objective Pathway Analysis: Accuracy (Green) vs. Effective Rank (Blue)", fontsize=18, pad=25)
    plt.ylabel("Normalized Range [0-1]", fontsize=13)
    
    # Custom Legend
    legend_elements = [
        Line2D([0], [0], color="#2ecc71", lw=3, alpha=0.7, label='Top 1% Accuracy'),
        Line2D([0], [0], color="#3498db", lw=3, alpha=0.7, label='Top 1% Effective Rank'),
        Line2D([0], [0], color="#710d0d", lw=1, alpha=0.3, label='Baseline Runs')
    ]
    plt.legend(handles=legend_elements, loc='upper right', fontsize=11)
    
    plt.grid(axis='y', linestyle='--', alpha=0.2)
    plt.grid(axis='x', linestyle='-', alpha=0.1)
    plt.tight_layout()
    plt.show()

# Run the dual analysis
plot_combined_pathways(df)

In [ ]:
def plot_bifurcation_sweep(df):
    # 1. Sort by H_INERTIA to see the progression
    df_sweep = df.sort_values('H_INERTIA')
    
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Plot Accuracy on the first Y-axis
    color_acc = '#2ecc71'
    ax1.set_xlabel('H_INERTIA (The Master Parameter)', fontsize=12)
    ax1.set_ylabel('Accuracy', color=color_acc, fontsize=12)
    # Using a rolling mean to show the trend through the noise
    ax1.plot(df_sweep['H_INERTIA'], df_sweep['acc'].rolling(window=10).mean(), 
             color=color_acc, linewidth=3, label='Accuracy Trend')
    ax1.scatter(df_sweep['H_INERTIA'], df_sweep['acc'], color=color_acc, alpha=0.1, s=10)
    ax1.tick_params(axis='y', labelcolor=color_acc)

    # Create a second Y-axis for Effective Rank
    ax2 = ax1.twinx()
    color_rank = '#3498db'
    ax2.set_ylabel('Effective Rank (Complexity)', color=color_rank, fontsize=12)
    ax2.plot(df_sweep['H_INERTIA'], df_sweep['rank'].rolling(window=10).mean(), 
             color=color_rank, linewidth=3, linestyle='--', label='Rank Trend')
    ax2.tick_params(axis='y', labelcolor=color_rank)

    plt.title('Bifurcation Analysis: The Dominant Effect of H_INERTIA', fontsize=15, pad=20)
    ax1.grid(True, alpha=0.2)
    
    # Adding a vertical line at the "Critical Point"
    # (Assuming the peak accuracy happens at a specific inertia)
    optimal_h = df.loc[df['acc'].idxmax(), 'H_INERTIA']
    plt.axvline(x=optimal_h, color='red', linestyle=':', alpha=0.5, label='Optimal H')

    fig.tight_layout()
    plt.show()

plot_bifurcation_sweep(df)

In [ ]:
def plot_synchrony_crash(df):
    df_sweep = df.sort_values('H_INERTIA')
    
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # Plot Accuracy (The Goal)
    color_acc = '#2ecc71' # Green
    ax1.set_xlabel('H_INERTIA', fontsize=12)
    ax1.set_ylabel('Accuracy', color=color_acc, fontsize=12)
    ax1.plot(df_sweep['H_INERTIA'], df_sweep['acc'].rolling(window=15).mean(), 
             color=color_acc, linewidth=3, label='Accuracy')
    ax1.tick_params(axis='y', labelcolor=color_acc)

    # Plot Synchrony (The Physical State)
    ax2 = ax1.twinx()
    color_sync = '#e67e22' # Orange
    ax2.set_ylabel('Global Synchrony', color=color_sync, fontsize=12)
    ax2.plot(df_sweep['H_INERTIA'], df_sweep['sync'].rolling(window=15).mean(), 
             color=color_sync, linewidth=3, linestyle='-.', label='Synchrony')
    ax2.tick_params(axis='y', labelcolor=color_sync)

    plt.title('The Criticality Threshold: When High Synchrony Kills Accuracy', fontsize=15, pad=20)
    ax1.grid(True, alpha=0.2)
    
    # Highlight the "Goldilocks Zone"
    plt.axvspan(0.85, 0.93, color='yellow', alpha=0.1, label='Functional Regime')

    fig.tight_layout()
    plt.show()

plot_synchrony_crash(df)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_rank_sync_h_inertia(df):
    # 1. Sort by H_INERTIA to visualize the transition
    df_sweep = df.sort_values('H_INERTIA')
    
    fig, ax1 = plt.subplots(figsize=(12, 6))

    # --- PRIMARY Y-AXIS: EFFECTIVE RANK ---
    color_rank = '#3498db' # Blue
    ax1.set_xlabel('H_INERTIA', fontsize=12)
    ax1.set_ylabel('Effective Rank (Dimensionality)', color=color_rank, fontsize=12)
    
    # We use a rolling mean (window=15) to see the trend through the Sobol noise
    ax1.plot(df_sweep['H_INERTIA'], df_sweep['rank'].rolling(window=15).mean(), 
             color=color_rank, linewidth=3, label='Rank (Structural Complexity)')
    ax1.scatter(df_sweep['H_INERTIA'], df_sweep['rank'], color=color_rank, alpha=0.1, s=10)
    ax1.tick_params(axis='y', labelcolor=color_rank)

    # --- SECONDARY Y-AXIS: SYNCHRONY ---
    ax2 = ax1.twinx()
    color_sync = '#e67e22' # Orange
    ax2.set_ylabel('Global Synchrony (Binding)', color=color_sync, fontsize=12)
    
    ax2.plot(df_sweep['H_INERTIA'], df_sweep['sync'].rolling(window=15).mean(), 
             color=color_sync, linewidth=3, linestyle='-.', label='Synchrony')
    ax2.tick_params(axis='y', labelcolor=color_sync)

    # --- FORMATTING ---
    plt.title('The Complexity-Synchrony Trade-off in Global Fields', fontsize=15, pad=20)
    ax1.grid(True, alpha=0.2)
    
    # Highlight the "Critical Window" where Rank is high but Synchrony is still rising
    # Adjust these numbers based on your specific results (0.85-0.93 was your sweet spot)
    plt.axvspan(0.85, 0.93, color='grey', alpha=0.1, label='Metastable Regime')

    # Combined Legend
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, loc='upper left')

    fig.tight_layout()
    plt.show()

# Run the analysis
plot_rank_sync_h_inertia(df)

In [ ]:
import seaborn as sns

def plot_gf_distribution_sweep(df):
    plt.figure(figsize=(10, 6))
    
    # This creates a "Ridge Plot" feel by showing how the 
    # Global Synchrony (the GF's output) spreads out.
    sns.kdeplot(data=df, x="H_INERTIA", y="sync", 
                cmap="magma", fill=True, thresh=0, levels=30)
    
    plt.title("The GF State Transition Map")
    plt.xlabel("H_INERTIA (Inertia)")
    plt.ylabel("Global Synchrony (Field Strength)")
    plt.show()
plot_gf_distribution_sweep(df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_acc_landscape_sweep(df):
    plt.figure(figsize=(10, 6))
    
    # We use 'viridis' or 'plasma' for accuracy to distinguish it from the 'magma' sync plot
    # This creates a "Topographical Map" of your model's performance
    sns.kdeplot(data=df, x="H_INERTIA", y="acc", 
                cmap="viridis", fill=True, thresh=0, levels=30)
    
    # Overlay individual runs to see the outliers/sobol noise
    plt.scatter(df['H_INERTIA'], df['acc'], color='white', s=1, alpha=0.2)
    
    plt.title("Functional Performance Landscape (Accuracy vs. Inertia)", fontsize=14)
    plt.xlabel("H_INERTIA (Global Field Inertia)")
    plt.ylabel("Accuracy")
    
    # Vertical lines to mark the "Functional Cliff" you discovered
    plt.axvline(x=0.93, color='red', linestyle='--', alpha=0.5, label='Bifurcation Point')
    
    plt.grid(True, alpha=0.1)
    plt.show()

plot_acc_landscape_sweep(df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_rank_landscape_sweep(df):
    plt.figure(figsize=(10, 6))
    
    # We'll use 'plasma' or 'magma' here to highlight the intensity of complexity
    sns.kdeplot(data=df, x="H_INERTIA", y="rank", 
                cmap="plasma", fill=True, thresh=0, levels=30)
    
    # Scatter overlay to see the Sobol sampling points
    plt.scatter(df['H_INERTIA'], df['rank'], color='white', s=1, alpha=0.15)
    
    plt.title("Structural Complexity Landscape (Rank vs. Inertia)", fontsize=14)
    plt.xlabel("H_INERTIA (Stickiness)")
    plt.ylabel("Effective Rank (System Dimensionality)")
    
    # Mark the collapse point
    plt.axvline(x=0.93, color='cyan', linestyle='--', alpha=0.6, label='Complexity Collapse')
    
    plt.grid(True, alpha=0.1)
    plt.show()

plot_rank_landscape_sweep(df)

In [ ]:
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import time
import os
import os
import tensorflow as tf
from tensorflow.keras import mixed_precision
import gc
import tensorflow as tf
import numpy as np
from tqdm import tqdm
import scipy.linalg
import math
import os, gc, random
import matplotlib.pyplot as plt
import json
import pickle
import time
from datetime import datetime
from IPython.display import clear_output

tf.keras.mixed_precision.set_global_policy('float32')

# Note: Your model will now output float16. 
# Ensure your last layer is float32 for stability:
# self.out = tf.keras.layers.Dense(num_classes, dtype='float32')

# Standard Conda path for libdevice
conda_cuda_path = os.path.join(os.environ.get('CONDA_PREFIX', ''), "Library", "bin")
if os.path.exists(conda_cuda_path):
    os.environ['XLA_FLAGS'] = f"--xla_gpu_cuda_data_dir={conda_cuda_path}"
    print(f"XLA Path set to: {conda_cuda_path}")

# EMERGENCY BYPASS: If it still fails, disable JIT for now. 
# It will be slightly slower, but it will NOT crash.
USE_JIT = False

import os
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir="C:/Users/caspe/anaconda3/envs/SPIKEDETEC/Library/bin"'

# 1. Load MNIST
(x_all, y_all), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# 2. Flatten and Normalize (Standard step)
x_all = x_all.reshape(-1, 784).astype('uint8') / 255.0
x_test = x_test.reshape(-1, 784).astype('uint8') / 255.0

# 3. Create FIXED Permutation (The "p" in pMNIST)
rng = np.random.RandomState(42)
perm = rng.permutation(784)

x_all = x_all[:, perm]
x_test = x_test[:, perm]

# --- NEW: Train/Val Split (90% Train, 10% Val) ---
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(
    x_all, y_all, test_size=0.1, random_state=42, stratify=y_all
)

# 4. Reshape to [Batch, Time, Channels] for your RNN
x_train = x_train[:, :, np.newaxis]
x_val   = x_val[:, :, np.newaxis]
x_test  = x_test[:, :, np.newaxis]

# 5. Labels to int32
y_train = y_train.astype('int32')
y_val   = y_val.astype('int32')
y_test  = y_test.astype('int32')

# 6. Build datasets
train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).shuffle(10000).batch(256)
val_ds   = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(256)
test_ds  = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(256)

print(f"pMNIST Ready. Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")

import tensorflow as tf
# --- 1. CONFIGURATION ---


# Forcefully disable the strict determinism flag at the TF level
try:
    tf.config.experimental.enable_op_determinism(False)
    print("Success: Strict determinism disabled.")
except:
    print("Warning: Could not toggle flag. If Phase 1 fails, restart kernel.")
def reset_seeds():
    import os
    import gc
    import random
    import numpy as np

    # 1. Clear session
    tf.keras.backend.clear_session()
    
    # 2. Disable strict determinism to allow GPU kernels to run
    os.environ['TF_DETERMINISTIC_OPS'] = '0' 
    
    np.random.seed(42)
    tf.random.set_seed(42)
  
    np.random.seed(42)
    tf.random.set_seed(42)
    np.random.seed(42)
    tf.random.set_seed(42)
  
  
    # 3. Hard-lock all seeds
    os.environ['PYTHONHASHSEED'] = str(42)
    random.seed(42)
    np.random.seed(42)
    tf.random.set_seed(42)
    tf.keras.utils.set_random_seed(42) 
    
    gc.collect()
    print("Environment Reset: Seeds locked at 42. Strict determinism disabled.")
    
def print_param_report(hidden_size, input_dim=1, num_classes=10):
    """
    Calculates and prints a detailed breakdown of parameters for the 
    OscillatingResonator architecture.
    """
    # RNN Layer 1: Inputs from data
    rnn1_params = (input_dim * hidden_size) + (hidden_size * hidden_size) + hidden_size
    
    # RNN Layer 2 & 3: Inputs from previous hidden layer
    rnn_mid_params = (hidden_size * hidden_size) + (hidden_size * hidden_size) + hidden_size
    
    # Final Dense Layer
    dense_params = (hidden_size * num_classes) + num_classes
    
    total = rnn1_params + (2 * rnn_mid_params) + dense_params
    
    print(f"--- PARAMETER COUNT REPORT (Hidden: {hidden_size}) ---")
    print(f"RNN Layer 1: {rnn1_params:>8,}")
    print(f"RNN Layer 2: {rnn_mid_params:>8,}")
    print(f"RNN Layer 3: {rnn_mid_params:>8,}")
    print(f"Dense Out:   {dense_params:>8,}")
    print("-" * 35)
    print(f"TOTAL PARAMS: {total:>8,}")
    print(f"Estimated Memory: {total * 4 / 1024:.2f} KB (float32)")
    print("=" * 35 + "\n")

from sklearn.metrics import confusion_matrix

def plot_resonator_confusion(model, x_test_full, y_test_full, data_percent=DATA_PERCENT, reversed_mode=False, session_name="Session"):
    # 1. Respect the data constraint
    num_test = int(len(x_test_full) * data_percent)
    x_subset = x_test_full[:num_test]
    y_true = y_test_full[:num_test]

    # 2. Handle Reverse Mode (Internal labeling only)
    title_prefix = "REVERSED" if reversed_mode else "STANDARD"
    if reversed_mode:
        x_subset = x_subset[:, ::-1, :]

    # 3. Get predictions
    print(f"--> Predicting for {title_prefix}...")
    logits, _ = model.predict(x_subset, batch_size=BATCH_SIZE, verbose=0)
    y_pred = np.argmax(logits, axis=-1)
    y_true_np = y_true.numpy() if hasattr(y_true, 'numpy') else y_true

    # 4. Build ASCII Confusion Matrix
    cm = confusion_matrix(y_true_np, y_pred, labels=range(10))
    
    cm_lines = []
    cm_lines.append(f"\n{'#'*70}")
    cm_lines.append(f" {title_prefix} EVALUATION | SESSION: {session_name}")
    cm_lines.append(f"{'#'*70}")
    cm_lines.append("Pred:    0    1    2    3    4    5    6    7    8    9")
    cm_lines.append("True |" + "----" * 12)
    
    for i, row in enumerate(cm):
        row_str = f"{i:>3}  | " + " ".join([f"{val:>4}" for val in row])
        cm_lines.append(row_str)
    
    cm_lines.append("-" * 60)
    
    # 5. Calculate Bias Score
    most_common_idx = np.argmax(np.sum(cm, axis=0))
    bias_pct = (np.sum(cm[:, most_common_idx]) / np.sum(cm)) * 100
    cm_lines.append(f"Primary Bias: Digit '{most_common_idx}' accounts for {bias_pct:.1f}% of all predictions.")
    cm_lines.append(f"{'#'*70}\n")
    
    ascii_matrix = "\n".join(cm_lines)

    # 6. APPEND to the main session log ONLY (stripping any stray prefixes)
    # This ensures it goes into log_RES_H16...txt even if called weirdly
    clean_session = session_name.replace("Standard_", "").replace("Reversed_", "")
    log_filename = f"log_{clean_session}.txt"
    
    with open(log_filename, "a") as f:
        f.write(ascii_matrix)
    
    # Still print to console for the live check
    print(ascii_matrix)

def plot_field_jitter(model, sample_batch):
    # 1. Extract hidden sequences
    logits, h_seq = model(sample_batch[:1], training=False)
    h_np = h_seq[0].numpy() # Shape: (784, HIDDEN)
    
    # 2. Reconstruct physics components
    base_delta = (2.0 * np.pi) / PERIOD
    steps = np.arange(h_np.shape[0])
    phase = steps * base_delta
    
    # Raw Oscillator and Neuronal Bias
    ghost_field = np.sin(phase)
    activity_bias = JITTER_SCALE * np.mean(h_np, axis=-1) 
    combined_signal = ghost_field + activity_bias
    
    # Reconstruct G_norm (Simplified EMA for visualization)
    # This approximates the 'new_G' EMA in your cell logic
    ema_g = 0
    g_history = []
    for step_h in h_np:
        # Shuffling the signal as per your 'half' concat logic
        half = len(step_h) // 2
        shuffled = np.concatenate([step_h[half:], step_h[:half]])
        ema_g = (1.0 - LAMBDA_SLOW) * ema_g + LAMBDA_SLOW * shuffled
        g_history.append(ema_g)
    
    g_history = np.array(g_history)
    # Z-score normalization for the tanh gating
    g_norm = (g_history - np.mean(g_history)) / (np.std(g_history) + 1e-6)
    
    # Calculate Final Gain Modulation (Field Effect)
    # FE = 1 + (strength * combined * tanh(G_norm))
    # We take the mean across neurons for the final visualization
    field_effect = 1.0 + (BASE_STRENGTH * combined_signal[:, None] * np.tanh(g_norm))
    field_effect_mean = np.mean(field_effect, axis=-1)

    # 3. Plotting
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    
    # Plot 1: Interference Pattern
    axes[0].plot(ghost_field, label="Pure Sine (Ghost)", color='gray', alpha=0.4, linestyle='--')
    axes[0].plot(combined_signal, label="Combined (Active)", color='#3498db', linewidth=1.5)
    axes[0].set_title("Carrier vs. Signal Interference")
    axes[0].legend()

    # Plot 2: Neuronal Bias (The DC Offset)
    axes[1].axhline(0, color='black', lw=1, alpha=0.3)
    axes[1].fill_between(steps, activity_bias, color='#e74c3c', alpha=0.3, label="Neuronal Bias")
    axes[1].plot(activity_bias, color='#e74c3c', lw=2)
    axes[1].set_title(f"Hidden State Pressure (Scale: {JITTER_SCALE})")
    axes[1].legend()

    # Plot 3: Resulting Gain Modulation (The Field Effect)
    axes[2].axhline(1.0, color='black', lw=1, alpha=0.3) # 1.0 is neutral gain
    axes[2].plot(field_effect_mean, color='#2ecc71', lw=2, label="Field Effect (Gain)")
    axes[2].fill_between(steps, 1.0, field_effect_mean, color='#2ecc71', alpha=0.2)
    axes[2].set_title("Total Gain Modulation ($FE_t$)")
    axes[2].set_ylabel("Multiplicative Factor")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

# Run it!
sample_x, sample_y = next(iter(test_ds_subset))

def visualize_pmnist_complete(model, image, label, permutation):
    # 1. Setup Inverse Permutation
    inverse_perm = np.argsort(permutation)
    img_tensor = tf.convert_to_tensor(image[np.newaxis, ...], dtype=tf.float32)
    
    # 2. Manual hidden state extraction
    h1_seq = model.rnn1(img_tensor, training=False)
    h2_seq = model.rnn2(h1_seq, training=False)
    h3_seq = model.rnn3(h2_seq, training=False)
    
    # 3. Get Prediction and Gradients for Saliency
    with tf.GradientTape() as tape:
        tape.watch(img_tensor)        
        output = model(img_tensor, training=False)
        logits = output[0] if isinstance(output, tuple) else output
        
        # Calculate Prediction and Confidence
        probs = tf.nn.softmax(logits, axis=-1)
        pred_label = np.argmax(probs.numpy(), axis=-1)[0]
        confidence = np.max(probs.numpy(), axis=-1)[0]
        
        loss = logits[0, label]

    grads = tape.gradient(loss, img_tensor)
    saliency_flat = tf.reduce_max(tf.abs(grads), axis=-1).numpy().flatten()
    
    # 4. Prepare Images (Clean vs Scrambled)
    unscrambled_img = image.flatten()[inverse_perm].reshape(28, 28)
    unscrambled_sal = saliency_flat[inverse_perm].reshape(28, 28)
    scrambled_img = image.reshape(28, 28)
    scrambled_sal = saliency_flat.reshape(28, 28)

    # --- PLOTTING ---
    fig = plt.figure(figsize=(22, 12))
    
    # Color the title based on correctness
    result_color = "green" if pred_label == label else "red"
    
    # COLUMN 1: UNSCRAMBLED (The "Truth")
    ax1 = plt.subplot2grid((3, 5), (0, 0))
    ax1.imshow(unscrambled_img, cmap='gray')
    ax1.set_title(f"Target: {label} | PRED: {pred_label}\n({confidence*100:.1f}% Conf)", 
                  fontsize=14, color=result_color, fontweight='bold')
    
    ax2 = plt.subplot2grid((3, 5), (1, 0))
    ax2.imshow(unscrambled_sal, cmap='hot')
    ax2.set_title("Focus (Unscrambled)")

    # COLUMN 2: SCRAMBLED (The "Reality")
    ax3 = plt.subplot2grid((3, 5), (0, 1))
    ax3.imshow(scrambled_img, cmap='gray')
    ax3.set_title("Input (Scrambled)")
    
    ax4 = plt.subplot2grid((3, 5), (1, 1))
    ax4.imshow(scrambled_sal, cmap='hot')
    ax4.set_title("Focus (Scrambled)")

    # COLUMN 3-5: THE HIDDEN HEARTBEATS
    layers = [h1_seq, h2_seq, h3_seq]
    layer_names = ["Layer 1", "Layer 2", "Layer 3: Decision State"]
    
    for i, (h_seq, name) in enumerate(zip(layers, layer_names)):
        ax = plt.subplot2grid((3, 5), (i, 2), colspan=3)
        activity = tf.transpose(h_seq[0]).numpy()
        im = ax.imshow(activity, aspect='auto', cmap='magma', interpolation='nearest')
        ax.set_title(name)
        ax.set_ylabel("Neuron ID")
        if i == 2: ax.set_xlabel("Time (784 Steps)")
        plt.colorbar(im, ax=ax)

    plt.tight_layout()
    plt.show()

# Run it!
#visualize_pmnist_complete(model_active, x_test[101], y_test[101], perm)

from scipy.spatial.distance import euclidean
from scipy.stats import pearsonr

def correlate_jitter_fields(model, image_idx, x_data, period, strength, lambda_slow):
    # 1. Get Forward Jitter
    sample_fwd = x_data[image_idx:image_idx+1]
    _, h_fwd = model(sample_fwd, training=False)
    fe_fwd = calculate_field_effect(h_fwd[0].numpy(), period, strength, lambda_slow)

    # 2. Get Reversed Jitter
    sample_rev = sample_fwd[:, ::-1, :]
    _, h_rev = model(sample_rev, training=False)
    fe_rev = calculate_field_effect(h_rev[0].numpy(), period, strength, lambda_slow)
    
    # 3. Flip the Reversed signal back to compare 1:1 with Forward
    fe_rev_flipped = fe_rev[::-1]

    # 4. Math Metrics
    corr, _ = pearsonr(fe_fwd, fe_rev_flipped)
    dist = euclidean(fe_fwd, fe_rev_flipped)
    
    print(f"\n--- SYMMETRY ANALYSIS (Hash {image_idx}) ---")
    print(f"Correlation Score (R): {corr:.4f}  (1.0 = Perfect Symmetry)")
    print(f"Euclidean Distance:    {dist:.4f}  (0.0 = Identical Field)")
    
    return corr, dist

# Helper to match your plot_field_jitter logic
def calculate_field_effect(h_np, period, strength, lambda_slow):
    steps = np.arange(len(h_np))
    ghost = np.sin(steps * (2.0 * np.pi / period))
    bias = np.mean(h_np, axis=-1)
    combined = ghost + bias
    
    # Simple EMA to simulate G_norm
    ema_g = 0
    g_history = []
    for step_h in h_np:
        half = len(step_h) // 2
        shuffled = np.concatenate([step_h[half:], step_h[:half]])
        ema_g = (1.0 - lambda_slow) * ema_g + lambda_slow * shuffled
        g_history.append(ema_g)
    
    g_norm = (np.array(g_history) - np.mean(g_history)) / (np.std(g_history) + 1e-6)
    # Mean Field Effect across neurons
    fe = 1.0 + (strength * combined[:, None] * np.tanh(g_norm))
    return np.mean(fe, axis=-1)

from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

def analyze_symmetry(model, hash_std, hash_rev, x_data):
    # 1. Generate Fields
    def get_fe(idx, reversed=False):
        batch = x_data[idx:idx+1]
        if reversed: batch = batch[:, ::-1, :]
        _, h_seq = model(batch, training=False)
        return calculate_field_effect(h_seq[0].numpy(), PERIOD, BASE_STRENGTH, LAMBDA_SLOW)

    fe_std = get_fe(hash_std, reversed=False)
    fe_rev = get_fe(hash_rev, reversed=True)
    
    # Flip the reversed field to align 1:1 with the standard timeline
    fe_rev_aligned = fe_rev[::-1]

    # 2. Math Metrics
    corr, _ = pearsonr(fe_std, fe_rev_aligned)
    dist = euclidean(fe_std, fe_rev_aligned)

    print(f"\n{'='*40}")
    print(f" FIELD SYMMETRY: Hash {hash_std} vs Hash {hash_rev}")
    print(f"{'='*40}")
    print(f" Pearson Correlation: {corr:.4f}  (Target: 1.0)")
    print(f" Euclidean Distance:  {dist:.4f}  (Target: 0.0)")
    print(f"{'='*40}\n")
    
    # Run your existing plots
    plot_field_jitter(model, x_data[hash_std:hash_std+1])
    plot_field_jitter(model, x_data[hash_rev:hash_rev+1][:, ::-1, :])
 
import numpy as np
import tensorflow as tf
from SALib.sample import saltelli
from SALib.analyze import sobol
import matplotlib.pyplot as plt
import gc
from datetime import datetime
import json
DATA_PERCENT = 0.01
BATCH_SIZE = 4 * 64
EPOCHS = 2         
HIDDEN = 32 
REST_BASELINE = 1.0
LEARNING_RATE = 1e-3
#Changed in sobol
BASE_STRENGTH =0
LAMBDA_SLOW = 0
PERIOD= 0 
JITTER_SCALE = 0
N_baseline = 8 # T
# --- 1.1 AUTO-GENERATED SESSION NAME ---
# Creates a name like: RES_H32_S0.40_J1.15_T123456
now = datetime.now()
readable_ts = now.strftime("%y%m%d_%H%M") 
SESSION_ID = f"RES_H{HIDDEN}_S{BASE_STRENGTH}_J{JITTER_SCALE}_{readable_ts}"
print(f" SESSION INITIALIZED: {SESSION_ID}")
print_param_report(HIDDEN)

sobol_problem = {
    'num_vars': 5,
    'names': ['LAMBDA_SLOW', 'H_INERTIA','BASE_STRENGTH', 'PERIOD', 'JITTER_SCALE'],
    'bounds': [
        [0.001, 0.05], 
        [0.01, 0.99],
        [0.01, 0.99],      
        [784/10, 784/2], 
        [0.1, 1.2]
    ]
}

def calc_sobol_samples(problem, N, second_order=True):
    """
    Calculates total runs for a SALib Saltelli/Sobol sequence.
    Formula: N * (2D + 2) if second_order=True
             N * (D + 2)  if second_order=False
    """
    D = problem['num_vars']
    if second_order:
        total = N * (2 * D + 2)
    else:
        total = N * (D + 2)
    return total

# Your specific setup

total_runs = calc_sobol_samples(sobol_problem, N_baseline, second_order=True)

print(f"--- SOBOL RUN ESTIMATION ---")
print(f"Variables (D): {sobol_problem['num_vars']}")
print(f"Baseline (N):  {N_baseline}")
print(f"Total Model Evaluations: {total_runs}")

# Time estimation (Optional but helpful for dissertations)
avg_time_per_run = EPOCHS*24 # seconds (Estimate based on your Epochs)
total_hours = (total_runs * avg_time_per_run) / 3600
total_days = total_hours/24
print(f"Estimated Time: {total_hours:.2f} hours")
print(f"Estimated Time: {total_days:.2f} days")






# --- 2. THE JITTERED CELL ---
class JitteredFeedbackCell(tf.keras.layers.Layer):
    def __init__(self, units=16, strength=0.0, period=256.0, lambda_slow=0.05, mode="active", **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = [units, units, 1] 
        self.strength = strength 
        self.period = period
        self.lambda_slow = lambda_slow
        self.mode = mode

    def build(self, input_shape):
        self.w_in = self.add_weight(shape=(input_shape[-1], self.units), name="w_in", initializer="glorot_uniform")
        self.w_rec = self.add_weight(shape=(self.units, self.units), name="w_rec",
                                     initializer=tf.keras.initializers.Orthogonal(gain=1.0))
        self.bias = self.add_weight(shape=(self.units,), name="bias", initializer="zeros")

    def call(self, inputs, states):
        prev_h, prev_G, prev_phase = states
        half = self.units // 2
        raw_signal = tf.concat([prev_h[:, half:], prev_h[:, :half]], axis=1)
        source_signal = tf.stop_gradient(raw_signal) if self.mode == "probe" else raw_signal
        new_G = (1.0 - self.lambda_slow) * prev_G + self.lambda_slow * source_signal
        G_mean = tf.reduce_mean(new_G, axis=-1, keepdims=True)
        G_std = tf.math.reduce_std(new_G, axis=-1, keepdims=True) + 1e-6
        G_norm = (new_G - G_mean) / G_std
        new_phase = prev_phase + (2.0 * math.pi / self.period)
        oscillator = tf.math.sin(new_phase)
        
        if self.mode == "active":
            bias_signal = tf.reduce_mean(source_signal, axis=-1, keepdims=True) - 0.1
            combined_signal = oscillator + (JITTER_SCALE * bias_signal)
        else:
            combined_signal = oscillator

        current_strength = self.strength * combined_signal if self.mode != "passive" else 0.0
        field_effect = REST_BASELINE + (current_strength * tf.tanh(G_norm))
        z = (tf.matmul(inputs, self.w_in) + tf.matmul(prev_h, self.w_rec) + self.bias) * field_effect
        h = (H_SCALE[0] * prev_h) + (H_SCALE[1] * tf.nn.elu(z))
        h = tf.clip_by_value(h, -20.0, 20.0)
        return h, [h, new_G, new_phase]

class OscillatingResonator(tf.keras.Model):
    def __init__(self, hidden=16, num_classes=10, strength=0.0, mode="active"):
        super().__init__()
        self.cell_ref = JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode)
        self.rnn1 = tf.keras.layers.RNN(self.cell_ref, return_sequences=True)
        self.rnn2 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.rnn3 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.out = tf.keras.layers.Dense(num_classes)

    def call(self, x, training=False):
        h1 = self.rnn1(x, training=training)
        h2 = self.rnn2(h1, training=training)
        h3 = self.rnn3(h2, training=training)
        return self.out(h3[:, -1, :]), h3

# --- 3. UPDATED LOGGING UTILITY ---
def print_history_summary(history, model, model_name="Model", test_acc=None):
    cell = model.rnn1.cell
    total_params = model.count_params()
    
    print(f"\n DATA LOG: {model_name}")
    print(f" CONFIG: Hidden: {cell.units} | Params: {total_params:,} | F-Wgt: {cell.strength:.4f} | "
          f"L-Slow (τ): {cell.lambda_slow:.3f} | Jitter: {JITTER_SCALE:.2f} | Period: {cell.period:.1f} | Data: {DATA_PERCENT*100:.2f}%")
    print("="*145)
    header = f"{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6} | {'F-Wgt':<6}"
    print(header)
    print("-" * 145)

    for i in range(len(history['loss'])):
        m = history['hidden_metrics'][i]
        print(f"{i+1:<6} | {history['loss'][i]:<7.3f} | {history['acc'][i]*100:<8.2f} | "
              f"{m['effective_rank']:<6.2f} | {m['synchrony']:<6.3f} | {m['entropy']:<6.2f} | "
              f"{m['a_corr']:<7.3f} | {m['interference']:<6.3f} | {cell.strength:<6.3f}")
    
    print("-" * 145)
    test_str = f"{test_acc*100:.2f}%" if test_acc is not None else "N/A"
    print(f" FINAL PERFORMANCE: Validation Acc: {history['acc'][-1]*100:.2f}% | TEST ACCURACY: {test_str}")
    print("="*145 + "\n")

# --- 4. TRAINING PHASE WITH SNAPSHOT & LIVE LOGS ---
# --- UPDATED TRAINING PHASE ---
def train_phase(model, train_data, val_data, epochs=3, name="model"):
    current_lr = LEARNING_RATE
    optimizer = tf.keras.optimizers.Adam(learning_rate=current_lr)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    history = {"loss": [], "acc": [], "hidden_metrics": []}
    best_val_acc = 0.0

    @tf.function
    def train_step(x, y, lr):
        optimizer.learning_rate.assign(lr)
        with tf.GradientTape() as tape:
            logits, _ = model(x, training=True)
            loss_v = loss_fn(y, logits)
        grads = tape.gradient(loss_v, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss_v, logits

    print(f"\n{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7}")
    print("-" * 75)

    for epoch in range(epochs):
        acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
        epoch_loss = []
        pbar = tqdm(train_data, desc=f"EPOCH {epoch+1}/{epochs}", leave=False)
        
        for x_b, y_b in pbar:
            loss_v, logits = train_step(x_b, y_b, current_lr)
            acc_metric.update_state(y_b, logits)
            epoch_loss.append(float(loss_v))
            pbar.set_postfix({"loss": f"{np.mean(epoch_loss):.4f}", "acc": f"{acc_metric.result():.2%}"})

        # --- VALIDATION SNAPSHOT ---
        for x_v, y_v in val_data.take(1):
            logits_v, h_seq_v = model(x_v, training=False)
            val_acc = np.mean(np.argmax(logits_v.numpy(), axis=-1) == y_v.numpy())
            
            # 1. SAVE BEST WEIGHTS
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                model.save_weights(f"best_{name}_{SESSION_ID}.weights.h5")
                status_msg = f" (New Best!)"
            else:
                status_msg = ""
            
            # Metric Calculations
            h_final = h_seq_v.numpy()[:, -1, :]
            s = scipy.linalg.svdvals(h_final) + 1e-12
            p_rank = s / (np.sum(s) + 1e-10)
            eff_rank = np.exp(-np.sum(p_rank * np.log(p_rank + 1e-10)))
            
            counts, _ = np.histogram(h_final, bins=50)
            p_ent = counts / (h_final.size + 1e-10) 
            entropy_val = -np.sum(p_ent * np.log2(p_ent + 1e-10))
            
            sync_val = (np.sum(np.abs(np.corrcoef(h_final.T + 1e-8))) - HIDDEN) / (HIDDEN**2 - HIDDEN)
            acorr_val = np.mean(np.abs(np.corrcoef(h_seq_v.numpy()[0].T + 1e-8)))


            h_final = h_seq_v.numpy()[:, -1, :]
            pop_mean = np.mean(h_final, axis=1, keepdims=True)
            correlations = [
                np.corrcoef(h_final[:, i], pop_mean[:, 0])[0, 1]
                for i in range(h_final.shape[1])
            ]
            interference = np.mean(np.abs(correlations))

            history["loss"].append(np.mean(epoch_loss))
            history["acc"].append(val_acc)
            history["hidden_metrics"].append({
                "effective_rank": eff_rank,
                "synchrony": sync_val,
                "entropy": entropy_val,
                "a_corr": acorr_val,
                "interference": interference
            })

            print(f"{epoch+1:<6} | {np.mean(epoch_loss):<7.3f} | {val_acc*100:<8.2f} | "
                  f"{eff_rank:<6.2f} | {sync_val:<6.3f} | {entropy_val:<6.2f} | {acorr_val:<7.3f}{status_msg}")
    
    # 2. SAVE LATEST WEIGHTS (Post-training)
    model.save_weights(f"latest_{name}_{SESSION_ID}.weights.h5")
    print(f"--- Finished {name}: Best and Latest weights saved. ---")
            
    return history

# --- 5. SAVE ---
# This map will hold everything: configs, histories, and final scores
master_results = {
    "config": {
        "hidden": HIDDEN,
        "base_strength": BASE_STRENGTH,
        "jitter": JITTER_SCALE,
        "period": PERIOD,
        "epochs": EPOCHS,
        "data_pct": DATA_PERCENT
    },
    "runs": {}
}

def safe_evaluate(model, dataset):
    """Evaluates accuracy in batches to prevent GPU OOM errors."""
    accs = []
    for x_batch, y_batch in dataset:
        logits, _ = model(x_batch, training=False)
        preds = np.argmax(logits.numpy(), axis=-1)
        accs.append(np.mean(preds == y_batch.numpy()))
    return np.mean(accs)


# --- 6. EXECUTION ---
def reset_env():
    tf.keras.backend.clear_session()
    random.seed(42); np.random.seed(42); tf.random.set_seed(42)
    gc.collect()

# SETUP DATA
num_train = int(len(x_train) * DATA_PERCENT)
num_val = int(len(x_val)*DATA_PERCENT)
num_test  = int(len(x_test) * DATA_PERCENT)

train_ds = tf.data.Dataset.from_tensor_slices((x_train[:num_train], y_train[:num_train])) \
    .shuffle(5000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val[:1000], y_val[:1000])) \
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds_subset = tf.data.Dataset.from_tensor_slices((x_test[:num_test], y_test[:num_test])) \
    .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Test Run Ready: Samples -> Train: {num_train}, Val: {num_val}, Data %: {DATA_PERCENT*100}")

# --- RUN 1: ACTIVE ---
# --- 5. EXECUTION & UNIQUE SAVING ---

master_results = {
    "session_id": SESSION_ID,
    "config": {
        "hidden": HIDDEN, "base_strength": BASE_STRENGTH,
        "jitter": JITTER_SCALE, "period": PERIOD, "epochs": EPOCHS
    },
    "runs": {}
}

def save_metrics_to_files(history, model, model_name, test_acc):
    cell = model.rnn1.cell
    total_params = model.count_params()
    
    # 1. GENERATE THE TEXT TABLE 
    log_lines = []
    log_lines.append(f"\n DATA LOG: {model_name}")
    log_lines.append(f" CONFIG: Hidden: {cell.units} | Params: {total_params:,} | F-Wgt: {cell.strength:.4f} | "
                     f"L-Slow (τ): {cell.lambda_slow:.3f} | Jitter: {JITTER_SCALE:.2f} | Period: {cell.period:.1f} | Data: {DATA_PERCENT*100:.2f}%")
    log_lines.append("="*145)
    header = f"{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6} | {'F-Wgt':<6}"
    log_lines.append(header)
    log_lines.append("-" * 145)

    for i in range(len(history['loss'])):
        m = history['hidden_metrics'][i]
        line = (f"{i+1:<6} | {history['loss'][i]:<7.3f} | {history['acc'][i]*100:<8.2f} | "
                f"{m['effective_rank']:<6.2f} | {m['synchrony']:<6.3f} | {m['entropy']:<6.2f} | "
                f"{m['a_corr']:<7.3f} | {m['interference']:<6.3f} | {cell.strength:<6.3f}")
        log_lines.append(line)
    
    log_lines.append("-" * 145)
    test_str = f"{test_acc*100:.2f}%" if test_acc is not None else "N/A"
    log_lines.append(f" FINAL PERFORMANCE: Validation Acc: {history['acc'][-1]*100:.2f}% | TEST ACCURACY: {test_str}")
    log_lines.append("="*145 + "\n")

    # Save to TXT (Appends so you don't lose previous runs)
    with open(f"log_{SESSION_ID}.txt", "a") as f:
        f.write("\n".join(log_lines))
    
    # Print to console as usual
    print("\n".join(log_lines))

    # 2. SAVE RAW DATA TO JSON
    json_data = {
        "session_id": SESSION_ID,
        "model_name": model_name,
        "config": {
            "hidden": cell.units, "strength": cell.strength, "tau": cell.lambda_slow,
            "jitter": JITTER_SCALE, "period": PERIOD, "data_pct": DATA_PERCENT
        },
        "history": history,
        "test_accuracy": float(test_acc)
    }
    
    # Simple conversion for numpy types
    def clean_types(obj):
        if isinstance(obj, dict): return {k: clean_types(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean_types(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64)): return float(obj)
        if isinstance(obj, (np.int32, np.int64)): return int(obj)
        return obj

    with open(f"results_{SESSION_ID}_{model_name}.json", "w") as f:
        json.dump(clean_types(json_data), f, indent=4)

import tensorflow as tf
import numpy as np
import scipy.linalg
import json
import gc
import math
from tqdm.auto import tqdm

# --- 1. GLOBAL MASTER LOGGING ---
SOBOL_MASTER_DATA = {"session_id": SESSION_ID, "runs": {}, "Si_all": {}}

def update_json_log(history, model, run_id):
    """Saves every epoch's progress into one master 'Big Ass' JSON file."""
    global SOBOL_MASTER_DATA
    if run_id is None: return # Skip if not a Sobol run
    
    cell = model.rnn1.cell
    SOBOL_MASTER_DATA["runs"][str(run_id)] = {
        "config": {
            "tau": float(cell.lambda_slow),
            "strength": float(cell.strength),
            "jitter": float(JITTER_SCALE),
            "period": float(cell.period),
            #"learning rate": float(LEARNING_RATE),
            "h_inertia": float(H_SCALE[0])
        },
        "history": history
    }

    def clean(obj):
        if isinstance(obj, dict): return {k: clean(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64, np.ndarray)): 
            return float(obj) if np.isscalar(obj) else obj.tolist()
        return obj

    with open(f"SOBOL_MASTER_{SESSION_ID}.json", "w") as f:
        json.dump(clean(SOBOL_MASTER_DATA), f, indent=4)

# --- 2. THE JITTERED CELL & MODEL ---
class JitteredFeedbackCell(tf.keras.layers.Layer):
    def __init__(self, units=16, strength=0.0, period=256.0, lambda_slow=0.05, mode="active", **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.state_size = [units, units, 1] 
        self.strength = strength 
        self.period = period
        self.lambda_slow = lambda_slow
        self.mode = mode

    def build(self, input_shape):
        self.w_in = self.add_weight(shape=(input_shape[-1], self.units), name="w_in", initializer="glorot_uniform")
        self.w_rec = self.add_weight(shape=(self.units, self.units), name="w_rec",
                                     initializer=tf.keras.initializers.Orthogonal(gain=1.0))
        self.bias = self.add_weight(shape=(self.units,), name="bias", initializer="zeros")

    def call(self, inputs, states):
        prev_h, prev_G, prev_phase = states
        half = self.units // 2
        raw_signal = tf.concat([prev_h[:, half:], prev_h[:, :half]], axis=1)
        source_signal = tf.stop_gradient(raw_signal) if self.mode == "probe" else raw_signal
        
        new_G = (1.0 - self.lambda_slow) * prev_G + self.lambda_slow * source_signal
        G_norm = (new_G - tf.reduce_mean(new_G, axis=-1, keepdims=True)) / (tf.math.reduce_std(new_G, axis=-1, keepdims=True) + 1e-6)
        
        new_phase = prev_phase + (2.0 * math.pi / self.period)
        oscillator = tf.math.sin(new_phase)
        
        if self.mode == "active":
            bias_signal = tf.reduce_mean(source_signal, axis=-1, keepdims=True) - 0.1
            combined_signal = oscillator + (JITTER_SCALE * bias_signal)
        else:
            combined_signal = oscillator

        current_strength = self.strength * combined_signal if self.mode != "passive" else 0.0
        field_effect = REST_BASELINE + (current_strength * tf.tanh(G_norm))
        
        z = (tf.matmul(inputs, self.w_in) + tf.matmul(prev_h, self.w_rec) + self.bias) * field_effect
        h = (H_SCALE[0] * prev_h) + (H_SCALE[1] * tf.nn.elu(z))
        h = tf.clip_by_value(h, -20.0, 20.0)
        return h, [h, new_G, new_phase]

class OscillatingResonator(tf.keras.Model):
    def __init__(self, hidden=16, num_classes=10, strength=0.0, mode="active"):
        super().__init__()
        # Use cell_ref so we can access hyperparameters easily later
        self.cell_ref = JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode)
        self.rnn1 = tf.keras.layers.RNN(self.cell_ref, return_sequences=True)
        self.rnn2 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.rnn3 = tf.keras.layers.RNN(JitteredFeedbackCell(hidden, strength, PERIOD, LAMBDA_SLOW, mode=mode), return_sequences=True)
        self.out = tf.keras.layers.Dense(num_classes)

    def call(self, x, training=False):
        h1 = self.rnn1(x, training=training)
        h2 = self.rnn2(h1, training=training)
        h3 = self.rnn3(h2, training=training)
        return self.out(h3[:, -1, :]), h3

# --- 3. UPDATED TRAINING PHASE ---
def train_phase(model, train_data, val_data, epochs=3, name="model", run_id=None):
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    history = {"loss": [], "acc": [], "hidden_metrics": []}
    best_val_acc = 0.0

    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            logits, _ = model(x, training=True)
            loss_v = loss_fn(y, logits)
        grads = tape.gradient(loss_v, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss_v, logits

    # Added 'Intf' to the console header
    print(f"\n{'Epoch':<6} | {'Loss':<7} | {'Val-Acc%':<8} | {'Rank':<6} | {'Sync':<6} | {'Entrp':<6} | {'A-Corr':<7} | {'Intf':<6}")
    print("-" * 85)

    for epoch in range(epochs):
        epoch_losses = []
        acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
        pbar = tqdm(train_data, desc=f"EPOCH {epoch+1}/{epochs}", leave=False,disable=True)
        
        for x_b, y_b in pbar:
            loss_v, logits = train_step(x_b, y_b)
            acc_metric.update_state(y_b, logits)
            epoch_losses.append(float(loss_v))
            pbar.set_postfix({"loss": f"{np.mean(epoch_losses):.4f}", "acc": f"{acc_metric.result():.2%}"})

        # --- VALIDATION SNAPSHOT ---
        for x_v, y_v in val_data.take(1):
            logits_v, h_seq_v = model(x_v, training=False)
            val_acc = np.mean(np.argmax(logits_v.numpy(), axis=-1) == y_v.numpy())            
            
            # --- METRIC CALCULATIONS ---
            h_final = h_seq_v.numpy()[:, -1, :]
            
            # 1. Effective Rank
            s = scipy.linalg.svdvals(h_final) + 1e-12
            p_rank = s / (np.sum(s) + 1e-10)
            eff_rank = np.exp(-np.sum(p_rank * np.log(p_rank + 1e-10)))
            
            # 2. Entropy
            counts, _ = np.histogram(h_final, bins=50)
            p_ent = counts / (h_final.size + 1e-10) 
            entropy_val = -np.sum(p_ent * np.log2(p_ent + 1e-10))
            
            # 3. Synchrony (Inter-neuron correlation)
            sync_val = (np.sum(np.abs(np.corrcoef(h_final.T + 1e-8))) - HIDDEN) / (HIDDEN**2 - HIDDEN)
            
            # 4. Auto-Correlation (Temporal slowness)
            acorr_val = np.mean(np.abs(np.corrcoef(h_seq_v.numpy()[0].T + 1e-8)))

            # 5. NEW: Interference (Neuron-to-Field alignment)
            # Measures how much neurons are driven by the common field vs unique input
            mean_field = np.mean(h_final, axis=1, keepdims=True)
            neuron_to_field_corrs = [
                np.abs(np.corrcoef(h_final[:, j], mean_field[:, 0])[0, 1]) 
                for j in range(h_final.shape[1])
            ]
            interference_val = np.mean(np.nan_to_num(neuron_to_field_corrs))

        # --- HISTORY UPDATE ---
        avg_loss = np.mean(epoch_losses)
        history["loss"].append(float(avg_loss))
        history["acc"].append(float(val_acc))
        history["hidden_metrics"].append({
            "effective_rank": float(eff_rank),
            "synchrony": float(sync_val),
            "entropy": float(entropy_val),
            "a_corr": float(acorr_val),
            "interference": float(interference_val)
        })

        # --- THE BIG JSON SAVE ---
        update_json_log(history, model, run_id)
        
        # Unified Console Output
        print(f"EP {epoch+1}/{epochs}: {avg_loss:<7.3f} | {val_acc:<8.2%} | {eff_rank:<6.2f} | "
              f"{sync_val:<6.3f} | {entropy_val:<6.2f} | {acorr_val:<7.3f} | {interference_val:<6.3f}")

    return history


# --- 7. SOBOL SENSITIVITY ANALYSIS IMPLEMENTATION (FIXED) ---

SOBOL_MASTER_DATA = {
    "session_id": SESSION_ID,
    "problem": sobol_problem,
    "runs": {},
    "Si_all": {} 
}

def evaluate_sobol_run(params, run_id):
    # Map back to the globals the model uses
    global LEARNING_RATE, JITTER_SCALE, BASE_STRENGTH, PERIOD, LAMBDA_SLOW, H_INERTIA, H_SCALE
    
    current_lslow, h_inertia, current_strength, current_period, current_jitter = params
    
    LAMBDA_SLOW = current_lslow
    H_INERTIA = h_inertia          
    BASE_STRENGTH = current_strength
    PERIOD = current_period
    JITTER_SCALE = current_jitter
    
    # Critical derivation for architectural parity
    H_SCALE = [H_INERTIA, 1.0 - H_INERTIA] 

    # Clear memory from previous run to prevent VRAM accumulation
    tf.keras.backend.clear_session()
    gc.collect()
    
    # Build model (Will use the updated globals)
    # Reset seeds here for identical initialization across configurations
    tf.keras.utils.set_random_seed(42)
    model = OscillatingResonator(hidden=HIDDEN, num_classes=10, strength=BASE_STRENGTH, mode="active")
    
    # Standard Header for the Run
    print(f"\n[Sobol Run {run_id}/{total_runs}] "
          f"H_Inertia: {H_INERTIA:.4f} | "
          f"BASE_S: {BASE_STRENGTH:.4f} | "
          f"Tau: {LAMBDA_SLOW:.4f} | "
          f"Jitter: {JITTER_SCALE:.2f}")

    # Execute training
    # Note: ensure train_phase uses print(f"\r...", end="", flush=True) for epoch updates
    history = train_phase(model, train_ds, val_ds, epochs=EPOCHS, name="sobol", run_id=run_id)
    
    # Final newline to clear the 'flush' line from train_phase
    print("") 

    # Safely extract metrics from the final epoch
    metrics = {
        "acc": float(history['acc'][-1]),
        "rank": float(history['hidden_metrics'][-1]['effective_rank']),
        "sync": float(history['hidden_metrics'][-1]['synchrony']),
        "entr": float(history['hidden_metrics'][-1]['entropy']),
        "acorr": float(history['hidden_metrics'][-1]['a_corr']),
        "Intf": float(history['hidden_metrics'][-1]['interference'])
    }
    
    # Cleanup model reference
    del model
    clear_output(wait=True)
    return metrics

def run_sobol_analysis():
    print(f"--- STARTING MULTI-METRIC SOBOL ANALYSIS ({SESSION_ID}) ---")
    
    # SALib generates N * (2D + 2) samples
    # For D=5, N=8, this results in 8 * (10 + 2) = 96 runs
    param_values = saltelli.sample(sobol_problem, N_baseline, calc_second_order=True) 
    num_runs = len(param_values)
    
    metric_keys = ["acc", "rank", "sync", "entr", "acorr", "Intf"]
    Y = {m: np.zeros(num_runs) for m in metric_keys}

    for i, params in enumerate(param_values):
        try:
            res = evaluate_sobol_run(params, run_id=i)
            for key in metric_keys: 
                Y[key][i] = res[key]
        except Exception as e:
            # Move to next line so the error doesn't overwrite the progress line
            print(f"\n!! Run {i} failed: {e}")
            # Fill with NaN so the analyzer can handle the missing data
            for key in metric_keys: Y[key][i] = np.nan 

    # Calculate Sensitivity Indices for all tracked metrics
    Si_all = {}
    for key in metric_keys:
        y_clean = Y[key]
        nan_mask = np.isnan(y_clean)
        n_failed = np.sum(nan_mask)
        
        if n_failed > 0:
            print(f"Warning: {n_failed} failed runs for metric '{key}'")
        
        # If more than 10% of runs failed, the indices might be unreliable
        if n_failed / len(y_clean) > 0.1:
            print(f"Skipping {key}: failure rate too high ({n_failed}/{len(y_clean)})")
            Si_all[key] = None
            continue
            
        # Perform Sobol Analysis
        Si = sobol.analyze(sobol_problem, y_clean, print_to_console=False)
        Si_all[key] = {
            "S1": Si['S1'].tolist(),
            "ST": Si['ST'].tolist(),
            "S2": Si['S2'].tolist() if 'S2' in Si else None,
            "n_failed": int(n_failed)
        }

    SOBOL_MASTER_DATA["Si_all"] = Si_all
    
    # JSON Serialization helper for Numpy/Tensorflow types
    def clean(obj):
        if isinstance(obj, dict): return {k: clean(v) for k, v in obj.items()}
        if isinstance(obj, list): return [clean(i) for i in obj]
        if isinstance(obj, (np.float32, np.float64, np.ndarray)): 
            return float(obj) if np.isscalar(obj) else obj.tolist()
        return obj

    output_path = f"SOBOL_MASTER_{SESSION_ID}.json"
    with open(output_path, "w") as f:
        json.dump(clean(SOBOL_MASTER_DATA), f, indent=4)
        
    print(f"\nFINISH. Full sensitivity results saved to: {output_path}")
    return Si_all

# --- EXECUTION ---
Si_all = run_sobol_analysis()




[Sobol Run 7/96] H_Inertia: 0.4694 | BASE_S: 0.8369 | Tau: 0.0485 | Jitter: 0.27

Epoch  | Loss    | Val-Acc% | Rank   | Sync   | Entrp  | A-Corr  | Intf  
-------------------------------------------------------------------------------------


KeyboardInterrupt: 

: 